In [0]:
spark

In [0]:
from pyspark.sql.types import *

# Define schema for StudentData
df1_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("salary", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True)
])

# Define schema for StudentData2
df2_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("salary", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True)
])

# Read first CSV file
df1 = spark.read.format("csv") \
    .schema(df1_schema) \
    .option("header", "true") \
    .load("/Volumes/workspace/default/volume1/sales.csv")

# Read second CSV file
df2 = spark.read.format("csv") \
    .schema(df2_schema) \
    .option("header", "true") \
    .load("/Volumes/workspace/default/volume1/sales.csv")

# Display both DataFrames
df1.show()
df2.show()

+----+----------+----+------+---------------+--------------+-----+---------+
|  id|      name| age|salary|         gender|          city|state|  country|
+----+----------+----+------+---------------+--------------+-----+---------+
|NULL|2025-12-31|NULL|  NULL|    Electronics|    Headphones|    5| 96575.33|
|NULL|2024-04-27|NULL|  NULL|Home Appliances|Vacuum Cleaner|    1|  1906.09|
|NULL|2025-09-22|NULL|  NULL|Home Appliances|    Air Cooler|    1|  2846.41|
|NULL|2023-03-09|NULL|  NULL|       Clothing|        Jacket|    3|  5702.72|
|NULL|2025-05-07|NULL|  NULL|       Clothing|         Shirt|    7|  8081.14|
|NULL|2026-07-28|NULL|  NULL|         Beauty|       Perfume|    4|  5616.23|
|NULL|2024-05-18|NULL|  NULL|    Electronics|    Smartphone|    3| 74878.39|
|NULL|2025-02-04|NULL|  NULL|Home Appliances|Vacuum Cleaner|    7| 41855.59|
|NULL|2025-09-15|NULL|  NULL|      Furniture|         Table|    3| 11614.55|
|NULL|2025-11-06|NULL|  NULL|    Electronics|       Monitor|    2| 34954.73|

In [0]:

from pyspark.sql import functions as F
from functools import reduce

# ==========================================
# SHOPPULSE GOLD LAYER
# STEP 1.2 - DATE RANGE ANALYSIS
# ==========================================

# Identify date columns from Silver tables
date_sources = [
    ("silver_orders", "order_purchase_timestamp"),
    ("silver_orders", "order_approved_at"),
    ("silver_orders", "order_delivered_carrier_date"),
    ("silver_orders", "order_delivered_customer_date"),
    ("silver_orders", "order_estimated_delivery_date"),

    ("silver_reviews", "review_creation_date"),
    ("silver_reviews", "review_answer_timestamp"),

    ("silver_marketing_qualified_leads", "first_contact_date"),

    ("silver_closed_deals", "won_date")
]

# Collect event dates into a common DataFrame
date_dfs = []

for table_name, column_name in date_sources:

    df = spark.table(
        f"workspace.silver.{table_name}"
    )

    date_df = df.select(
        F.to_date(
            F.col(column_name)
        ).alias("event_date")
    )

    date_dfs.append(date_df)

# Combine all date sources
all_event_dates_df = reduce(
    lambda df1, df2: df1.unionByName(df2),
    date_dfs
)

# Calculate earliest and latest dates
date_range = all_event_dates_df.agg(
    F.min("event_date").alias("earliest_date"),
    F.max("event_date").alias("latest_date"),
    F.count("event_date").alias("total_date_records")
).first()

print("========== SHOPPULSE DATE RANGE ==========")

print("Earliest Date:", date_range["earliest_date"])
print("Latest Date:", date_range["latest_date"])
print("Total Non-NULL Date Records:", date_range["total_date_records"])

# Count distinct calendar dates
print(
    "Distinct Calendar Dates:",
    all_event_dates_df
    .select("event_date")
    .distinct()
    .count()
)


========== SHOPPULSE DATE RANGE ==========
Earliest Date: 2016-09-04
Latest Date: 2018-11-14
Total Non-NULL Date Records: 698144
Distinct Calendar Dates: 757


In [0]:

from pyspark.sql import functions as F

# ==========================================
# SHOPPULSE GOLD LAYER
# STEP 1.4 - CREATE DATE DIMENSION
# ==========================================

# 1. Get date boundaries from previous profiling
date_bounds = all_event_dates_df.agg(
    F.min("event_date").alias("start_date"),
    F.max("event_date").alias("end_date")
).first()

start_date = date_bounds["start_date"]
end_date = date_bounds["end_date"]

# 2. Calculate total calendar days
total_days = (end_date - start_date).days + 1

print("Calendar Start Date:", start_date)
print("Calendar End Date:", end_date)
print("Total Calendar Days:", total_days)

# 3. Generate continuous date sequence
date_spine_df = (
    spark.range(total_days)
    .select(
        F.date_add(
            F.lit(start_date).cast("date"),
            F.col("id").cast("int")
        ).alias("full_date")
    )
)

# 4. Create date dimension attributes
gold_dim_date_df = (
    date_spine_df

    .withColumn(
        "date_key",
        F.date_format("full_date", "yyyyMMdd").cast("int")
    )

    .withColumn(
        "year",
        F.year("full_date")
    )

    .withColumn(
        "quarter",
        F.concat(
            F.lit("Q"),
            F.quarter("full_date").cast("string")
        )
    )

    .withColumn(
        "year_quarter",
        F.concat(
            F.year("full_date").cast("string"),
            F.lit("-Q"),
            F.quarter("full_date").cast("string")
        )
    )

    .withColumn(
        "month_number",
        F.month("full_date")
    )

    .withColumn(
        "month_name",
        F.date_format("full_date", "MMMM")
    )

    .withColumn(
        "year_month",
        F.date_format("full_date", "yyyy-MM")
    )

    .withColumn(
        "day_of_month",
        F.dayofmonth("full_date")
    )

    .withColumn(
        "day_name",
        F.date_format("full_date", "EEEE")
    )

    .withColumn(
        "day_of_week",
        F.dayofweek("full_date")
    )

    .withColumn(
        "day_of_year",
        F.dayofyear("full_date")
    )

    .withColumn(
        "week_of_year",
        F.weekofyear("full_date")
    )

    .withColumn(
        "is_weekend",
        F.dayofweek("full_date").isin(1, 7)
    )

    .select(
        "date_key",
        "full_date",
        "year",
        "quarter",
        "year_quarter",
        "month_number",
        "month_name",
        "year_month",
        "day_of_month",
        "day_name",
        "day_of_week",
        "day_of_year",
        "week_of_year",
        "is_weekend"
    )
    .orderBy("full_date")
)

print("\nGold Date Dimension created successfully!")

print(
    "Total Date Dimension Records:",
    gold_dim_date_df.count()
)

print(
    "Total Columns:",
    len(gold_dim_date_df.columns)
)

# 5. Inspect schema
gold_dim_date_df.printSchema()

# 6. Display sample records
display(gold_dim_date_df.limit(15))


Calendar Start Date: 2016-09-04
Calendar End Date: 2018-11-14
Total Calendar Days: 802

Gold Date Dimension created successfully!
Total Date Dimension Records: 802
Total Columns: 14
root
 |-- date_key: integer (nullable = true)
 |-- full_date: date (nullable = false)
 |-- year: integer (nullable = false)
 |-- quarter: string (nullable = false)
 |-- year_quarter: string (nullable = false)
 |-- month_number: integer (nullable = false)
 |-- month_name: string (nullable = false)
 |-- year_month: string (nullable = false)
 |-- day_of_month: integer (nullable = false)
 |-- day_name: string (nullable = false)
 |-- day_of_week: integer (nullable = false)
 |-- day_of_year: integer (nullable = false)
 |-- week_of_year: integer (nullable = false)
 |-- is_weekend: boolean (nullable = false)



date_key,full_date,year,quarter,year_quarter,month_number,month_name,year_month,day_of_month,day_name,day_of_week,day_of_year,week_of_year,is_weekend
20160904,2016-09-04,2016,Q3,2016-Q3,9,September,2016-09,4,Sunday,1,248,35,true
20160905,2016-09-05,2016,Q3,2016-Q3,9,September,2016-09,5,Monday,2,249,36,false
20160906,2016-09-06,2016,Q3,2016-Q3,9,September,2016-09,6,Tuesday,3,250,36,false
20160907,2016-09-07,2016,Q3,2016-Q3,9,September,2016-09,7,Wednesday,4,251,36,false
20160908,2016-09-08,2016,Q3,2016-Q3,9,September,2016-09,8,Thursday,5,252,36,false
20160909,2016-09-09,2016,Q3,2016-Q3,9,September,2016-09,9,Friday,6,253,36,false
20160910,2016-09-10,2016,Q3,2016-Q3,9,September,2016-09,10,Saturday,7,254,36,true
20160911,2016-09-11,2016,Q3,2016-Q3,9,September,2016-09,11,Sunday,1,255,36,true
20160912,2016-09-12,2016,Q3,2016-Q3,9,September,2016-09,12,Monday,2,256,37,false
20160913,2016-09-13,2016,Q3,2016-Q3,9,September,2016-09,13,Tuesday,3,257,37,false


In [0]:

# ==========================================
# STEP 1.5 - VALIDATE DATE DIMENSION
# ==========================================

# Total records
print(
    "Total Date Records:",
    gold_dim_date_df.count()
)

# Unique date keys
print(
    "Unique Date Keys:",
    gold_dim_date_df.select("date_key").distinct().count()
)

# Duplicate date keys
print(
    "Duplicate Date Keys:",
    gold_dim_date_df.count() -
    gold_dim_date_df.select("date_key").distinct().count()
)

# Check missing dates in continuous calendar
date_stats = gold_dim_date_df.agg(
    F.min("full_date").alias("minimum_date"),
    F.max("full_date").alias("maximum_date")
).first()

expected_days = (
    date_stats["maximum_date"] -
    date_stats["minimum_date"]
).days + 1

print("Expected Calendar Days:", expected_days)
print("Actual Calendar Days:", gold_dim_date_df.count())

# Check NULL values
null_count = gold_dim_date_df.filter(
    F.col("date_key").isNull() |
    F.col("full_date").isNull() |
    F.col("year").isNull() |
    F.col("month_name").isNull() |
    F.col("quarter").isNull()
).count()

print("Records with NULL Values:", null_count)

# Check weekend calculation
invalid_weekend_flags = gold_dim_date_df.filter(
    F.col("is_weekend") !=
    F.dayofweek("full_date").isin(1, 7)
).count()

print("Invalid Weekend Flags:", invalid_weekend_flags)

# Check calendar continuity
distinct_dates = (
    gold_dim_date_df
    .select("full_date")
    .distinct()
    .count()
)

print("Distinct Calendar Dates:", distinct_dates)


Total Date Records: 802
Unique Date Keys: 802
Duplicate Date Keys: 0
Expected Calendar Days: 802
Actual Calendar Days: 802
Records with NULL Values: 0
Invalid Weekend Flags: 0
Distinct Calendar Dates: 802


In [0]:

# Save Gold Date Dimension as Delta

gold_dim_date_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_date")

print("Successfully created: workspace.gold.dim_date")


Successfully created: workspace.gold.dim_date


In [0]:

# Load saved Gold Date Dimension
final_dim_date_df = spark.table(
    "workspace.gold.dim_date"
)

# Verify record count
print("Gold Date Records:", final_dim_date_df.count())

# Verify unique date keys
print(
    "Unique Date Keys:",
    final_dim_date_df
    .select("date_key")
    .distinct()
    .count()
)

# Verify column count
print(
    "Gold Date Columns:",
    len(final_dim_date_df.columns)
)

# Display schema
final_dim_date_df.printSchema()

# Verify Delta storage
display(
    spark.sql("""
        DESCRIBE DETAIL workspace.gold.dim_date
    """).select(
        "format",
        "name",
        "numFiles",
        "sizeInBytes"
    )
)

# Preview records
display(final_dim_date_df.limit(10))


Gold Date Records: 802
Unique Date Keys: 802
Gold Date Columns: 14
root
 |-- date_key: integer (nullable = true)
 |-- full_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: string (nullable = true)
 |-- year_quarter: string (nullable = true)
 |-- month_number: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- year_month: string (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- day_of_year: integer (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)



format,name,numFiles,sizeInBytes
delta,workspace.gold.dim_date,1,6678


date_key,full_date,year,quarter,year_quarter,month_number,month_name,year_month,day_of_month,day_name,day_of_week,day_of_year,week_of_year,is_weekend
20160904,2016-09-04,2016,Q3,2016-Q3,9,September,2016-09,4,Sunday,1,248,35,true
20160905,2016-09-05,2016,Q3,2016-Q3,9,September,2016-09,5,Monday,2,249,36,false
20160906,2016-09-06,2016,Q3,2016-Q3,9,September,2016-09,6,Tuesday,3,250,36,false
20160907,2016-09-07,2016,Q3,2016-Q3,9,September,2016-09,7,Wednesday,4,251,36,false
20160908,2016-09-08,2016,Q3,2016-Q3,9,September,2016-09,8,Thursday,5,252,36,false
20160909,2016-09-09,2016,Q3,2016-Q3,9,September,2016-09,9,Friday,6,253,36,false
20160910,2016-09-10,2016,Q3,2016-Q3,9,September,2016-09,10,Saturday,7,254,36,true
20160911,2016-09-11,2016,Q3,2016-Q3,9,September,2016-09,11,Sunday,1,255,36,true
20160912,2016-09-12,2016,Q3,2016-Q3,9,September,2016-09,12,Monday,2,256,37,false
20160913,2016-09-13,2016,Q3,2016-Q3,9,September,2016-09,13,Tuesday,3,257,37,false


In [0]:
from pyspark.sql import functions as F

# Load Silver Customers
silver_customers_df = spark.table(
    "workspace.silver.silver_customers"
)

# Check total records
print(
    "Silver Customers Records:",
    silver_customers_df.count()
)

# Check columns and data types
print("Silver Customers Columns:", len(silver_customers_df.columns))

silver_customers_df.printSchema()

# Display sample records
display(silver_customers_df.limit(10))

# Check unique customer IDs
print(
    "Unique Customer IDs:",
    silver_customers_df
    .select("customer_id")
    .distinct()
    .count()
)

Silver Customers Records: 99441
Silver Customers Columns: 7
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,_ingested_at,_source_file
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,2026-11-13T06:29:12.323Z,dbfs:/Volumes/workspace/default/shoppulse_raw/olist_customers_dataset.csv
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP,2026-11-13T06:29:12.323Z,dbfs:/Volumes/workspace/default/shoppulse_raw/olist_customers_dataset.csv
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP,2026-11-13T06:29:12.323Z,dbfs:/Volumes/workspace/default/shoppulse_raw/olist_customers_dataset.csv
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP,2026-11-13T06:29:12.323Z,dbfs:/Volumes/workspace/default/shoppulse_raw/olist_customers_dataset.csv
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,2026-11-13T06:29:12.323Z,dbfs:/Volumes/workspace/default/shoppulse_raw/olist_customers_dataset.csv
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC,2026-11-13T06:29:12.323Z,dbfs:/Volumes/workspace/default/shoppulse_raw/olist_customers_dataset.csv
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,04534,sao paulo,SP,2026-11-13T06:29:12.323Z,dbfs:/Volumes/workspace/default/shoppulse_raw/olist_customers_dataset.csv
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG,2026-11-13T06:29:12.323Z,dbfs:/Volumes/workspace/default/shoppulse_raw/olist_customers_dataset.csv
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR,2026-11-13T06:29:12.323Z,dbfs:/Volumes/workspace/default/shoppulse_raw/olist_customers_dataset.csv
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG,2026-11-13T06:29:12.323Z,dbfs:/Volumes/workspace/default/shoppulse_raw/olist_customers_dataset.csv


Unique Customer IDs: 99441


In [0]:

from pyspark.sql import functions as F

# ==========================================
# GOLD LAYER - CUSTOMER PROFILING
# ==========================================

# 1. Total records
total_records = silver_customers_df.count()

print("Total Customer Records:", total_records)


# 2. Check customer_id uniqueness
unique_customer_ids = (
    silver_customers_df
    .select("customer_id")
    .distinct()
    .count()
)

print("Unique Customer IDs:", unique_customer_ids)

print(
    "Duplicate Customer IDs:",
    total_records - unique_customer_ids
)


# 3. Check customer_unique_id uniqueness
unique_actual_customers = (
    silver_customers_df
    .select("customer_unique_id")
    .distinct()
    .count()
)

print("Unique Actual Customers:", unique_actual_customers)

print(
    "Repeated Customer Unique IDs:",
    total_records - unique_actual_customers
)


# 4. Check duplicate customer_id groups
duplicate_customer_ids = (
    silver_customers_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate Customer ID Groups:",
    duplicate_customer_ids.count()
)


# 5. NULL value analysis
customer_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]

print("\nNULL VALUE ANALYSIS:")

display(
    silver_customers_df.agg(
        *[
            F.count(
                F.when(F.col(c).isNull(), 1)
            ).alias(c)
            for c in customer_columns
        ]
    )
)


# 6. Empty string analysis
print("\nEMPTY STRING ANALYSIS:")

for c in [
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
]:
    count = silver_customers_df.filter(
        F.col(c).isNotNull() &
        (F.trim(F.col(c)) == "")
    ).count()

    print(f"Empty {c}: {count}")


# 7. ZIP code length validation
print("\nZIP CODE VALIDATION:")

print(
    "Invalid ZIP Length:",
    silver_customers_df.filter(
        F.length("customer_zip_code_prefix") != 5
    ).count()
)


# 8. State distribution
print("\nCUSTOMER STATE DISTRIBUTION:")

display(
    silver_customers_df
    .groupBy("customer_state")
    .count()
    .orderBy(F.desc("count"))
)


# 9. Customer geography summary
print("\nGEOGRAPHIC SUMMARY:")

display(
    silver_customers_df.agg(
        F.countDistinct("customer_city").alias("unique_cities"),
        F.countDistinct("customer_state").alias("unique_states"),
        F.countDistinct("customer_zip_code_prefix").alias("unique_zip_prefixes")
    )
)


Total Customer Records: 99441
Unique Customer IDs: 99441
Duplicate Customer IDs: 0
Unique Actual Customers: 96096
Repeated Customer Unique IDs: 3345
Duplicate Customer ID Groups: 0

NULL VALUE ANALYSIS:


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,0,0,0,0



EMPTY STRING ANALYSIS:
Empty customer_id: 0
Empty customer_unique_id: 0
Empty customer_city: 0
Empty customer_state: 0

ZIP CODE VALIDATION:
Invalid ZIP Length: 0

CUSTOMER STATE DISTRIBUTION:


customer_state,count
SP,41746
RJ,12852
MG,11635
RS,5466
PR,5045
SC,3637
BA,3380
DF,2140
ES,2033
GO,2020



GEOGRAPHIC SUMMARY:


unique_cities,unique_states,unique_zip_prefixes
4119,27,14994


In [0]:

from pyspark.sql import functions as F

# ==========================================
# SHOPPULSE GOLD LAYER
# STEP 2.4 - CREATE CUSTOMER DIMENSION
# ==========================================

# Select and standardize customer attributes
gold_dim_customers_df = (
    silver_customers_df

    .select(
        F.trim(F.col("customer_id")).alias("customer_id"),

        F.trim(
            F.col("customer_unique_id")
        ).alias("customer_unique_id"),

        F.lpad(
            F.col("customer_zip_code_prefix"),
            5,
            "0"
        ).alias("customer_zip_code_prefix"),

        F.trim(
            F.col("customer_city")
        ).alias("customer_city"),

        F.upper(
            F.trim(F.col("customer_state"))
        ).alias("customer_state")
    )
)

# Display transformation results
print("Gold Customer Dimension Created!")

print(
    "Original Records:",
    silver_customers_df.count()
)

print(
    "Gold Customer Records:",
    gold_dim_customers_df.count()
)

print(
    "Gold Customer Columns:",
    len(gold_dim_customers_df.columns)
)

# Inspect schema
gold_dim_customers_df.printSchema()

# Preview records
display(gold_dim_customers_df.limit(10))


Gold Customer Dimension Created!
Original Records: 99441
Gold Customer Records: 99441
Gold Customer Columns: 5
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,04534,sao paulo,SP
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


In [0]:

# ==========================================
# STEP 2.5 - CUSTOMER DIMENSION VALIDATION
# ==========================================

# 1. Total records
print(
    "Gold Customer Records:",
    gold_dim_customers_df.count()
)

# 2. Unique customer IDs
print(
    "Unique Customer IDs:",
    gold_dim_customers_df
    .select("customer_id")
    .distinct()
    .count()
)

# 3. Unique actual customers
print(
    "Unique Actual Customers:",
    gold_dim_customers_df
    .select("customer_unique_id")
    .distinct()
    .count()
)

# 4. Duplicate customer IDs
print(
    "Duplicate Customer IDs:",
    gold_dim_customers_df.count()
    - gold_dim_customers_df
    .select("customer_id")
    .distinct()
    .count()
)

# 5. Column count
print(
    "Gold Customer Columns:",
    len(gold_dim_customers_df.columns)
)

# 6. Invalid ZIP lengths
print(
    "Invalid ZIP Lengths:",
    gold_dim_customers_df.filter(
        F.length("customer_zip_code_prefix") != 5
    ).count()
)

# 7. NULL value analysis
print("NULL VALUE ANALYSIS:")

display(
    gold_dim_customers_df.agg(
        *[
            F.count(
                F.when(F.col(c).isNull(), 1)
            ).alias(c)
            for c in gold_dim_customers_df.columns
        ]
    )
)


Gold Customer Records: 99441
Unique Customer IDs: 99441
Unique Actual Customers: 96096
Duplicate Customer IDs: 0
Gold Customer Columns: 5
Invalid ZIP Lengths: 0
NULL VALUE ANALYSIS:


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,0,0,0,0


In [0]:

from pyspark.sql import functions as F

# ==========================================
# FINAL CUSTOMER DIMENSION VALIDATION
# ==========================================

# Check NULL values
print("NULL VALUE ANALYSIS:")

null_results = gold_dim_customers_df.agg(
    *[
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in gold_dim_customers_df.columns
    ]
).first().asDict()

for column, count in null_results.items():
    print(f"{column}: {count}")

# Check primary key validity
null_customer_ids = gold_dim_customers_df.filter(
    F.col("customer_id").isNull()
).count()

duplicate_customer_ids = (
    gold_dim_customers_df.count()
    - gold_dim_customers_df.select("customer_id").distinct().count()
)

# ==========================================
# SAVE GOLD CUSTOMER DIMENSION
# ==========================================

if (
    gold_dim_customers_df.count() == 99441
    and duplicate_customer_ids == 0
    and null_customer_ids == 0
):

    gold_dim_customers_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("workspace.gold.dim_customers")

    print("\nSuccessfully created: workspace.gold.dim_customers")

else:
    print("\nValidation failed. Gold table was not saved.")


# ==========================================
# VERIFY SAVED TABLE
# ==========================================

final_dim_customers_df = spark.table(
    "workspace.gold.dim_customers"
)

print("\n========== FINAL VERIFICATION ==========")

print(
    "Gold Customer Records:",
    final_dim_customers_df.count()
)

print(
    "Unique Customer IDs:",
    final_dim_customers_df.select("customer_id").distinct().count()
)

print(
    "Unique Actual Customers:",
    final_dim_customers_df.select("customer_unique_id").distinct().count()
)

print(
    "Gold Customer Columns:",
    len(final_dim_customers_df.columns)
)

final_dim_customers_df.printSchema()

# Verify Delta format
display(
    spark.sql("""
        DESCRIBE DETAIL workspace.gold.dim_customers
    """).select(
        "format",
        "name",
        "numFiles",
        "sizeInBytes"
    )
)


NULL VALUE ANALYSIS:
customer_id: 0
customer_unique_id: 0
customer_zip_code_prefix: 0
customer_city: 0
customer_state: 0

Successfully created: workspace.gold.dim_customers

========== FINAL VERIFICATION ==========
Gold Customer Records: 99441
Unique Customer IDs: 99441
Unique Actual Customers: 96096
Gold Customer Columns: 5
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



format,name,numFiles,sizeInBytes
delta,workspace.gold.dim_customers,1,3739035


In [0]:
from pyspark.sql import functions as F

# Load Silver Products
silver_products_df = spark.table(
    "workspace.silver.silver_products"
)

# Check total records
print(
    "Silver Products Records:",
    silver_products_df.count()
)

# Check columns
print(
    "Silver Products Columns:",
    len(silver_products_df.columns)
)

# Inspect schema
silver_products_df.printSchema()

# Display sample records
display(silver_products_df.limit(10))


Silver Products Records: 32951
Silver Products Columns: 10
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- product_name_length: integer (nullable = true)
 |-- product_description_length: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



product_id,product_category_name,product_category_name_english,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery,40,287,1,225,16,10,14
3aa071139cb16b67ca9e5dea641aaa2f,artes,art,44,276,1,1000,30,18,20
96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure,46,250,1,154,18,9,15
cef67bcfe19066a932b7673e239eb23d,bebes,baby,27,261,1,371,26,4,26
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares,37,402,4,625,20,17,13
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,musical_instruments,60,745,1,200,38,5,11
732bd381ad09e530fe0a5f457d81becb,cool_stuff,cool_stuff,56,1272,4,18350,70,24,44
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,furniture_decor,56,184,2,900,40,8,40
37cc742be07708b53a98702e77a21a02,eletrodomesticos,home_appliances,57,163,1,400,27,13,17
8c92109888e8cdf9d66dc7e463025574,brinquedos,toys,36,1156,1,600,17,10,12


In [0]:

from pyspark.sql import functions as F

# 1. Total products
total_products = silver_products_df.count()

print("Total Products:", total_products)


# 2. Unique product IDs
unique_products = (
    silver_products_df
    .select("product_id")
    .distinct()
    .count()
)

print("Unique Product IDs:", unique_products)

print(
    "Duplicate Product IDs:",
    total_products - unique_products
)


# 3. NULL value analysis
print("\nNULL VALUE ANALYSIS:")

display(
    silver_products_df.agg(
        *[
            F.count(
                F.when(F.col(c).isNull(), 1)
            ).alias(c)
            for c in silver_products_df.columns
        ]
    )
)


# 4. Product category distribution
print("\nPRODUCT CATEGORY ANALYSIS:")

display(
    silver_products_df
    .groupBy("product_category_name_english")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
)


# 5. Products without English categories
print(
    "Products Without English Category:",
    silver_products_df.filter(
        F.col("product_category_name_english").isNull()
    ).count()
)


# 6. Check invalid numeric attributes
numeric_columns = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for c in numeric_columns:
    print(
        f"Negative {c}:",
        silver_products_df.filter(
            F.col(c) < 0
        ).count()
    )


Total Products: 32951
Unique Product IDs: 32951
Duplicate Product IDs: 0

NULL VALUE ANALYSIS:


product_id,product_category_name,product_category_name_english,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,610,623,610,610,610,6,2,2,2



PRODUCT CATEGORY ANALYSIS:


product_category_name_english,count
bed_bath_table,3029
sports_leisure,2867
furniture_decor,2657
health_beauty,2444
housewares,2335
auto,1900
computers_accessories,1639
toys,1411
watches_gifts,1329
telephony,1134


Products Without English Category: 623
Negative product_name_length: 0
Negative product_description_length: 0
Negative product_photos_qty: 0
Negative product_weight_g: 0
Negative product_length_cm: 0
Negative product_height_cm: 0
Negative product_width_cm: 0


In [0]:

from pyspark.sql import functions as F

# ==========================================
# SHOPPULSE GOLD LAYER
# STEP 3.3 - CREATE PRODUCT DIMENSION
# ==========================================

# Load and select product attributes
gold_dim_products_df = (
    silver_products_df
    .select(
        F.trim(F.col("product_id")).alias("product_id"),
        "product_category_name",
        "product_category_name_english",
        "product_name_length",
        "product_description_length",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    )
)

print("Gold Product Dimension Created!")

print(
    "Original Records:",
    silver_products_df.count()
)

print(
    "Gold Product Records:",
    gold_dim_products_df.count()
)

print(
    "Gold Product Columns:",
    len(gold_dim_products_df.columns)
)

# Inspect schema
gold_dim_products_df.printSchema()

# Preview products
display(gold_dim_products_df.limit(10))


Gold Product Dimension Created!
Original Records: 32951
Gold Product Records: 32951
Gold Product Columns: 10
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- product_name_length: integer (nullable = true)
 |-- product_description_length: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



product_id,product_category_name,product_category_name_english,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery,40,287,1,225,16,10,14
3aa071139cb16b67ca9e5dea641aaa2f,artes,art,44,276,1,1000,30,18,20
96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure,46,250,1,154,18,9,15
cef67bcfe19066a932b7673e239eb23d,bebes,baby,27,261,1,371,26,4,26
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares,37,402,4,625,20,17,13
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,musical_instruments,60,745,1,200,38,5,11
732bd381ad09e530fe0a5f457d81becb,cool_stuff,cool_stuff,56,1272,4,18350,70,24,44
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,furniture_decor,56,184,2,900,40,8,40
37cc742be07708b53a98702e77a21a02,eletrodomesticos,home_appliances,57,163,1,400,27,13,17
8c92109888e8cdf9d66dc7e463025574,brinquedos,toys,36,1156,1,600,17,10,12


In [0]:

# ==========================================
# PRODUCT DIMENSION VALIDATION
# ==========================================

# 1. Total records
print(
    "Gold Product Records:",
    gold_dim_products_df.count()
)

# 2. Unique product IDs
print(
    "Unique Product IDs:",
    gold_dim_products_df
    .select("product_id")
    .distinct()
    .count()
)

# 3. Duplicate product IDs
print(
    "Duplicate Product IDs:",
    gold_dim_products_df.count()
    - gold_dim_products_df
    .select("product_id")
    .distinct()
    .count()
)

# 4. Column count
print(
    "Gold Product Columns:",
    len(gold_dim_products_df.columns)
)

# 5. Products without English categories
print(
    "Products Without English Category:",
    gold_dim_products_df.filter(
        F.col("product_category_name_english").isNull()
    ).count()
)

# 6. NULL Product IDs
print(
    "NULL Product IDs:",
    gold_dim_products_df.filter(
        F.col("product_id").isNull()
    ).count()
)

# 7. Negative numerical attributes
numeric_columns = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for c in numeric_columns:
    print(
        f"Negative {c}:",
        gold_dim_products_df.filter(
            F.col(c) < 0
        ).count()
    )

# 8. NULL value analysis
print("\nNULL VALUE ANALYSIS:")

display(
    gold_dim_products_df.agg(
        *[
            F.count(
                F.when(F.col(c).isNull(), 1)
            ).alias(c)
            for c in gold_dim_products_df.columns
        ]
    )
)


Gold Product Records: 32951
Unique Product IDs: 32951
Duplicate Product IDs: 0
Gold Product Columns: 10
Products Without English Category: 623
NULL Product IDs: 0
Negative product_name_length: 0
Negative product_description_length: 0
Negative product_photos_qty: 0
Negative product_weight_g: 0
Negative product_length_cm: 0
Negative product_height_cm: 0
Negative product_width_cm: 0

NULL VALUE ANALYSIS:


product_id,product_category_name,product_category_name_english,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,610,623,610,610,610,6,2,2,2


In [0]:
gold_dim_products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_products")

print("Successfully created: workspace.gold.dim_products")

Successfully created: workspace.gold.dim_products


In [0]:
final_dim_products_df = spark.table(
    "workspace.gold.dim_products"
)

print("Gold Product Records:", final_dim_products_df.count())

print(
    "Unique Product IDs:",
    final_dim_products_df.select("product_id").distinct().count()
)

print("Gold Product Columns:", len(final_dim_products_df.columns))

final_dim_products_df.printSchema()

Gold Product Records: 32951
Unique Product IDs: 32951
Gold Product Columns: 10
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- product_name_length: integer (nullable = true)
 |-- product_description_length: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [0]:
from pyspark.sql import functions as F

# Load Silver Sellers
silver_sellers_df = spark.table(
    "workspace.silver.silver_sellers"
)

# Check total records
print(
    "Silver Sellers Records:",
    silver_sellers_df.count()
)

# Check column count
print(
    "Silver Sellers Columns:",
    len(silver_sellers_df.columns)
)

# Inspect schema
silver_sellers_df.printSchema()

# Display sample records
display(silver_sellers_df.limit(10))

Silver Sellers Records: 3095
Silver Sellers Columns: 4
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



seller_id,seller_zip_code_prefix,seller_city,seller_state
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP
768a86e36ad6aae3d03ee3c6433d61df,01529,sao paulo,SP
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR


In [0]:

from pyspark.sql import functions as F

# 1. Total sellers
total_sellers = silver_sellers_df.count()

print("Total Sellers:", total_sellers)


# 2. Unique seller IDs
unique_seller_ids = (
    silver_sellers_df
    .select("seller_id")
    .distinct()
    .count()
)

print("Unique Seller IDs:", unique_seller_ids)

print(
    "Duplicate Seller IDs:",
    total_sellers - unique_seller_ids
)


# 3. NULL value analysis
print("\nNULL VALUE ANALYSIS:")

display(
    silver_sellers_df.agg(
        *[
            F.count(
                F.when(F.col(c).isNull(), 1)
            ).alias(c)
            for c in silver_sellers_df.columns
        ]
    )
)


# 4. ZIP-code validation
print("\nZIP CODE VALIDATION:")

print(
    "Invalid ZIP Length:",
    silver_sellers_df.filter(
        F.length("seller_zip_code_prefix") != 5
    ).count()
)


# 5. State distribution
print("\nSELLER STATE DISTRIBUTION:")

display(
    silver_sellers_df
    .groupBy("seller_state")
    .count()
    .orderBy(F.desc("count"))
)


# 6. Geographic summary
print("\nSELLER GEOGRAPHIC SUMMARY:")

display(
    silver_sellers_df.agg(
        F.countDistinct("seller_city").alias("unique_cities"),
        F.countDistinct("seller_state").alias("unique_states"),
        F.countDistinct("seller_zip_code_prefix").alias("unique_zip_prefixes")
    )
)


# 7. Empty string validation
for c in ["seller_id", "seller_city", "seller_state"]:
    print(
        f"Empty {c}:",
        silver_sellers_df.filter(
            F.col(c).isNotNull() &
            (F.trim(F.col(c)) == "")
        ).count()
    )



Total Sellers: 3095
Unique Seller IDs: 3095
Duplicate Seller IDs: 0

NULL VALUE ANALYSIS:


seller_id,seller_zip_code_prefix,seller_city,seller_state
0,0,0,0



ZIP CODE VALIDATION:
Invalid ZIP Length: 0

SELLER STATE DISTRIBUTION:


seller_state,count
SP,1849
PR,349
MG,244
SC,190
RJ,171
RS,129
GO,40
DF,30
ES,23
BA,19



SELLER GEOGRAPHIC SUMMARY:


unique_cities,unique_states,unique_zip_prefixes
611,23,2246


Empty seller_id: 0
Empty seller_city: 0
Empty seller_state: 0


In [0]:

from pyspark.sql import functions as F

# ==========================================
# SHOPPULSE GOLD LAYER
# STEP 4.3 - CREATE SELLER DIMENSION
# ==========================================

gold_dim_sellers_df = (
    silver_sellers_df
    .select(
        F.trim(F.col("seller_id")).alias("seller_id"),

        F.lpad(
            F.col("seller_zip_code_prefix"),
            5,
            "0"
        ).alias("seller_zip_code_prefix"),

        F.trim(
            F.col("seller_city")
        ).alias("seller_city"),

        F.upper(
            F.trim(F.col("seller_state"))
        ).alias("seller_state")
    )
)

print("Gold Seller Dimension Created!")

print(
    "Original Records:",
    silver_sellers_df.count()
)

print(
    "Gold Seller Records:",
    gold_dim_sellers_df.count()
)

print(
    "Gold Seller Columns:",
    len(gold_dim_sellers_df.columns)
)

gold_dim_sellers_df.printSchema()

display(gold_dim_sellers_df.limit(10))


Gold Seller Dimension Created!
Original Records: 3095
Gold Seller Records: 3095
Gold Seller Columns: 4
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



seller_id,seller_zip_code_prefix,seller_city,seller_state
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP
768a86e36ad6aae3d03ee3c6433d61df,01529,sao paulo,SP
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR


In [0]:

# ==========================================
# STEP 4.4 - SELLER VALIDATION AND SAVE
# ==========================================

# 1. Record count
total_records = gold_dim_sellers_df.count()

print("Gold Seller Records:", total_records)

# 2. Unique seller IDs
unique_sellers = (
    gold_dim_sellers_df
    .select("seller_id")
    .distinct()
    .count()
)

print("Unique Seller IDs:", unique_sellers)

# 3. Duplicate seller IDs
duplicate_sellers = total_records - unique_sellers

print("Duplicate Seller IDs:", duplicate_sellers)

# 4. Column count
print(
    "Gold Seller Columns:",
    len(gold_dim_sellers_df.columns)
)

# 5. ZIP length validation
invalid_zip = gold_dim_sellers_df.filter(
    F.length("seller_zip_code_prefix") != 5
).count()

print("Invalid ZIP Lengths:", invalid_zip)

# 6. NULL value analysis
print("\nNULL VALUE ANALYSIS:")

null_results = gold_dim_sellers_df.agg(
    *[
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in gold_dim_sellers_df.columns
    ]
).first().asDict()

for column, count in null_results.items():
    print(f"{column}: {count}")

total_nulls = sum(null_results.values())

# 7. Save only when validation passes
if (
    total_records == 3095
    and unique_sellers == 3095
    and duplicate_sellers == 0
    and len(gold_dim_sellers_df.columns) == 4
    and invalid_zip == 0
    and total_nulls == 0
):

    gold_dim_sellers_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("workspace.gold.dim_sellers")

    print("\nSuccessfully created: workspace.gold.dim_sellers")

else:
    print("\nValidation failed. Gold Sellers was not saved.")


# ==========================================
# STEP 4.5 - VERIFY SAVED GOLD TABLE
# ==========================================

if spark.catalog.tableExists("workspace.gold.dim_sellers"):

    final_dim_sellers_df = spark.table(
        "workspace.gold.dim_sellers"
    )

    print("\n========== FINAL VERIFICATION ==========")

    print(
        "Gold Seller Records:",
        final_dim_sellers_df.count()
    )

    print(
        "Unique Seller IDs:",
        final_dim_sellers_df
        .select("seller_id")
        .distinct()
        .count()
    )

    print(
        "Gold Seller Columns:",
        len(final_dim_sellers_df.columns)
    )

    final_dim_sellers_df.printSchema()

    display(
        spark.sql("""
            DESCRIBE DETAIL workspace.gold.dim_sellers
        """).select(
            "format",
            "name",
            "numFiles",
            "sizeInBytes"
        )
    )


Gold Seller Records: 3095
Unique Seller IDs: 3095
Duplicate Seller IDs: 0
Gold Seller Columns: 4
Invalid ZIP Lengths: 0

NULL VALUE ANALYSIS:
seller_id: 0
seller_zip_code_prefix: 0
seller_city: 0
seller_state: 0

Successfully created: workspace.gold.dim_sellers

========== FINAL VERIFICATION ==========
Gold Seller Records: 3095
Unique Seller IDs: 3095
Gold Seller Columns: 4
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



format,name,numFiles,sizeInBytes
delta,workspace.gold.dim_sellers,1,72225


In [0]:
from pyspark.sql import functions as F

# Load Silver Geolocation
silver_geolocation_df = spark.table(
    "workspace.silver.silver_geolocation"
)

# Check total records
print(
    "Silver Geolocation Records:",
    silver_geolocation_df.count()
)

# Check columns
print(
    "Silver Geolocation Columns:",
    len(silver_geolocation_df.columns)
)

# Inspect schema
silver_geolocation_df.printSchema()

# Preview records
display(silver_geolocation_df.limit(10))

Silver Geolocation Records: 738332
Silver Geolocation Columns: 5
root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)



geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
01036,-23.54131144157196,-46.64188583152521,sao paulo,SP
01007,-23.548339277829033,-46.63800653788448,sao paulo,SP
01027,-23.538015199549932,-46.631947411045076,são paulo,SP
01046,-23.546081127035535,-46.64482029837157,são paulo,SP
01030,-23.53652807216227,-46.633023105459124,sao paulo,SP
01022,-23.544295438253737,-46.632081127531876,sao paulo,SP
01040,-23.54160305551385,-46.639215518870124,sao paulo,SP
01027,-23.538015199549932,-46.631947411045076,sao paulo,SP
01037,-23.546723374689815,-46.6402811858157,sao paulo,SP
01050,-23.54891736303305,-46.642520665704815,são paulo,SP


In [0]:

# ==========================================
# GOLD LAYER - GEOLOCATION PROFILING
# ==========================================

# 1. Total geographic records
print(
    "Total Geolocation Records:",
    silver_geolocation_df.count()
)

# 2. Unique ZIP prefixes
unique_zip_prefixes = (
    silver_geolocation_df
    .select("geolocation_zip_code_prefix")
    .distinct()
    .count()
)

print(
    "Unique ZIP Prefixes:",
    unique_zip_prefixes
)

# 3. Profile geographic attributes per ZIP
zip_profile_df = (
    silver_geolocation_df
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.count("*").alias("geolocation_points"),

        F.countDistinct("geolocation_city")
        .alias("distinct_cities"),

        F.countDistinct("geolocation_state")
        .alias("distinct_states"),

        F.avg("geolocation_lat")
        .alias("average_latitude"),

        F.avg("geolocation_lng")
        .alias("average_longitude")
    )
)

print("\nZIP PROFILE SUMMARY:")

display(
    zip_profile_df.agg(
        F.count("*").alias("unique_zip_prefixes"),
        F.max("geolocation_points").alias("maximum_points_per_zip"),
        F.count(
            F.when(F.col("distinct_cities") > 1, 1)
        ).alias("zips_with_multiple_cities"),
        F.count(
            F.when(F.col("distinct_states") > 1, 1)
        ).alias("zips_with_multiple_states")
    )
)

# 4. Inspect ZIP prefixes associated with multiple cities or states
print("\nZIP PREFIXES WITH MULTIPLE LOCATIONS:")

display(
    zip_profile_df
    .filter(
        (F.col("distinct_cities") > 1) |
        (F.col("distinct_states") > 1)
    )
    .orderBy(F.desc("distinct_cities"))
    .limit(20)
)

# 5. Check customer and seller ZIP coverage
geo_zip_ref = (
    silver_geolocation_df
    .select("geolocation_zip_code_prefix")
    .distinct()
)

customer_zip_ref = (
    spark.table("workspace.gold.dim_customers")
    .select(
        F.col("customer_zip_code_prefix")
        .alias("geolocation_zip_code_prefix")
    )
    .distinct()
)

seller_zip_ref = (
    spark.table("workspace.gold.dim_sellers")
    .select(
        F.col("seller_zip_code_prefix")
        .alias("geolocation_zip_code_prefix")
    )
    .distinct()
)

missing_customer_zips = (
    customer_zip_ref
    .join(geo_zip_ref, "geolocation_zip_code_prefix", "left_anti")
    .count()
)

missing_seller_zips = (
    seller_zip_ref
    .join(geo_zip_ref, "geolocation_zip_code_prefix", "left_anti")
    .count()
)

print("\nGEOLOCATION REFERENCE VALIDATION:")
print("Missing Customer ZIP Prefixes:", missing_customer_zips)
print("Missing Seller ZIP Prefixes:", missing_seller_zips)


Total Geolocation Records: 738332
Unique ZIP Prefixes: 19015

ZIP PROFILE SUMMARY:


unique_zip_prefixes,maximum_points_per_zip,zips_with_multiple_cities,zips_with_multiple_states
19015,779,8555,8



ZIP PREFIXES WITH MULTIPLE LOCATIONS:


geolocation_zip_code_prefix,geolocation_points,distinct_cities,distinct_states,average_latitude,average_longitude
13455,76,5,1,-22.72668004322091,-47.36828329732867
06900,388,5,1,-23.835045093089278,-46.81412435296977
13457,42,5,1,-22.767759492049613,-47.38523368796597
13318,41,5,1,-23.282169355175753,-46.3031794246422
42850,187,5,1,-12.613335402339557,-38.30095545324635
28950,309,5,1,-22.766015618921784,-41.90626320952098
78290,28,5,1,-15.44906834303539,-58.737837914189825
17970,15,5,1,-21.26821380963202,-51.66655229000265
13454,120,5,1,-22.740573482065024,-47.3758887309465
45936,32,4,1,-14.767868686972276,-39.27809834669398



GEOLOCATION REFERENCE VALIDATION:
Missing Customer ZIP Prefixes: 157
Missing Seller ZIP Prefixes: 7


In [0]:

from pyspark.sql import functions as F

# Summarize geographic conflicts
geo_conflict_summary = zip_profile_df.agg(
    F.count("*").alias("total_zip_prefixes"),

    F.sum(
        F.when(F.col("distinct_cities") > 1, 1).otherwise(0)
    ).alias("zips_with_multiple_cities"),

    F.sum(
        F.when(F.col("distinct_states") > 1, 1).otherwise(0)
    ).alias("zips_with_multiple_states"),

    F.sum(
        F.when(
            (F.col("distinct_cities") > 1) |
            (F.col("distinct_states") > 1),
            1
        ).otherwise(0)
    ).alias("zips_with_location_conflicts"),

    F.max("geolocation_points").alias("maximum_points_per_zip")
).first()

print("Total ZIP Prefixes:", geo_conflict_summary["total_zip_prefixes"])
print("ZIPs with Multiple Cities:", geo_conflict_summary["zips_with_multiple_cities"])
print("ZIPs with Multiple States:", geo_conflict_summary["zips_with_multiple_states"])
print("ZIPs with Location Conflicts:", geo_conflict_summary["zips_with_location_conflicts"])
print("Maximum Points per ZIP:", geo_conflict_summary["maximum_points_per_zip"])

# Inspect conflicting ZIP prefixes
print("\nDetailed Location Conflicts:")

display(
    zip_profile_df
    .filter(
        (F.col("distinct_cities") > 1) |
        (F.col("distinct_states") > 1)
    )
    .orderBy(
        F.desc("distinct_states"),
        F.desc("distinct_cities")
    )
)


Total ZIP Prefixes: 19015
ZIPs with Multiple Cities: 8555
ZIPs with Multiple States: 8
ZIPs with Location Conflicts: 8558
Maximum Points per ZIP: 779

Detailed Location Conflicts:


geolocation_zip_code_prefix,geolocation_points,distinct_cities,distinct_states,average_latitude,average_longitude
72915,10,3,2,-15.391971910649469,-48.087266048564956
02116,11,2,2,-23.52035588962478,-46.585398449670926
80630,77,2,2,-25.48628118097366,-49.26743036313433
04011,70,2,2,-23.57949435790915,-46.64433461683851
78557,76,2,2,-11.87385144776555,-55.50065153233669
21550,145,1,2,-22.862106498127954,-43.356537433441275
79750,152,1,2,-22.24798269470084,-53.3473683429339
23056,31,1,2,-22.91815421885362,-43.6101346844442
13455,76,5,1,-22.72668004322091,-47.36828329732867
06900,388,5,1,-23.835045093089278,-46.81412435296977


In [0]:

from pyspark.sql import functions as F

# Eight ZIP prefixes with multiple states
state_conflict_zips = [
    "72915",
    "02116",
    "80630",
    "04011",
    "78557",
    "21550",
    "79750",
    "23056"
]

# Inspect actual city/state combinations
state_conflict_details_df = (
    silver_geolocation_df

    .withColumn(
        "zip_prefix",
        F.lpad(
            F.trim(F.col("geolocation_zip_code_prefix")),
            5,
            "0"
        )
    )

    .filter(
        F.col("zip_prefix").isin(state_conflict_zips)
    )

    .groupBy(
        "zip_prefix",
        "geolocation_city",
        "geolocation_state"
    )

    .agg(
        F.count("*").alias("geolocation_points"),

        F.avg("geolocation_lat").alias("average_latitude"),

        F.avg("geolocation_lng").alias("average_longitude")
    )

    .orderBy(
        "zip_prefix",
        F.desc("geolocation_points")
    )
)

print("Eight ZIP prefixes with multiple states:")

display(state_conflict_details_df)


Eight ZIP prefixes with multiple states:


zip_prefix,geolocation_city,geolocation_state,geolocation_points,average_latitude,average_longitude
02116,sao paulo,SP,7,-23.52060186141475,-46.58565247827886
02116,são paulo,SP,3,-23.521241202400475,-46.585881877107
02116,são paulo,RN,1,-23.515978148767935,-46.582169967107184
04011,sao paulo,SP,51,-23.579461992238475,-46.644175415615585
04011,são paulo,SP,18,-23.579629782238314,-46.64470546482431
04011,sao paulo,AC,1,-23.57870736918802,-46.645778615462866
21550,rio de janeiro,RJ,144,-22.862135977797173,-43.35656468860126
21550,rio de janeiro,AC,1,-22.85786142576025,-43.35261269040332
23056,rio de janeiro,RJ,30,-22.918120549114274,-43.61010260909802
23056,rio de janeiro,AC,1,-22.919164311034095,-43.61109694482953


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ==========================================
# SHOPPULSE GOLD LAYER
# STEP 5.6 - CREATE GEOLOCATION DIMENSION
# ==========================================

# STEP 1: Standardize source data

geo_base_df = (
    silver_geolocation_df
    .select(
        F.lpad(
            F.trim(F.col("geolocation_zip_code_prefix")),
            5,
            "0"
        ).alias("geolocation_zip_code_prefix"),

        F.col("geolocation_lat"),
        F.col("geolocation_lng"),

        F.lower(
            F.trim(F.col("geolocation_city"))
        ).alias("geolocation_city"),

        F.upper(
            F.trim(F.col("geolocation_state"))
        ).alias("geolocation_state")
    )
)


# STEP 2: Calculate ZIP-level statistics

geo_zip_summary_df = (
    geo_base_df
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.count("*").alias("geolocation_points"),

        F.countDistinct("geolocation_city")
        .alias("distinct_cities"),

        F.countDistinct("geolocation_state")
        .alias("distinct_states")
    )
)


# STEP 3: Identify the dominant state per ZIP

state_counts_df = (
    geo_base_df
    .groupBy(
        "geolocation_zip_code_prefix",
        "geolocation_state"
    )
    .agg(
        F.count("*").alias("state_point_count")
    )
)

state_window = (
    Window
    .partitionBy("geolocation_zip_code_prefix")
    .orderBy(
        F.desc("state_point_count"),
        F.asc("geolocation_state")
    )
)

dominant_state_df = (
    state_counts_df
    .withColumn(
        "state_rank",
        F.row_number().over(state_window)
    )
    .filter(F.col("state_rank") == 1)
    .select(
        "geolocation_zip_code_prefix",
        "geolocation_state"
    )
)


# STEP 4: Identify the dominant city within
# the selected dominant state

dominant_state_points_df = (
    geo_base_df
    .join(
        dominant_state_df,
        on=[
            "geolocation_zip_code_prefix",
            "geolocation_state"
        ],
        how="inner"
    )
)

city_counts_df = (
    dominant_state_points_df
    .groupBy(
        "geolocation_zip_code_prefix",
        "geolocation_city"
    )
    .agg(
        F.count("*").alias("city_point_count")
    )
)

city_window = (
    Window
    .partitionBy("geolocation_zip_code_prefix")
    .orderBy(
        F.desc("city_point_count"),
        F.asc("geolocation_city")
    )
)

dominant_city_df = (
    city_counts_df
    .withColumn(
        "city_rank",
        F.row_number().over(city_window)
    )
    .filter(F.col("city_rank") == 1)
    .select(
        "geolocation_zip_code_prefix",
        "geolocation_city"
    )
)


# STEP 5: Calculate representative coordinates
# using only the dominant city-state records

representative_points_df = (
    geo_base_df
    .join(
        dominant_state_df,
        on=[
            "geolocation_zip_code_prefix",
            "geolocation_state"
        ],
        how="inner"
    )
    .join(
        dominant_city_df,
        on=[
            "geolocation_zip_code_prefix",
            "geolocation_city"
        ],
        how="inner"
    )
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.avg("geolocation_lat").alias("geolocation_lat"),
        F.avg("geolocation_lng").alias("geolocation_lng")
    )
)


# STEP 6: Combine all ZIP-level attributes

geo_dimension_base_df = (
    geo_zip_summary_df

    .join(
        dominant_state_df,
        "geolocation_zip_code_prefix",
        "inner"
    )

    .join(
        dominant_city_df,
        "geolocation_zip_code_prefix",
        "inner"
    )

    .join(
        representative_points_df,
        "geolocation_zip_code_prefix",
        "inner"
    )

    .withColumn(
        "has_location_conflict",
        (
            (F.col("distinct_cities") > 1) |
            (F.col("distinct_states") > 1)
        )
    )
)


# STEP 7: Generate deterministic surrogate key

geo_key_window = Window.orderBy(
    "geolocation_zip_code_prefix"
)

gold_dim_geolocation_df = (
    geo_dimension_base_df
    .withColumn(
        "geo_key",
        F.row_number().over(geo_key_window).cast("long")
    )
    .select(
        "geo_key",
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state",
        "geolocation_points",
        "distinct_cities",
        "distinct_states",
        "has_location_conflict"
    )
)

print("Gold Geolocation Dimension Created!")

print(
    "Source Geolocation Records:",
    silver_geolocation_df.count()
)

print(
    "Unique ZIP Prefixes:",
    geo_zip_summary_df.count()
)

print(
    "Gold Geolocation Records:",
    gold_dim_geolocation_df.count()
)

print(
    "Gold Geolocation Columns:",
    len(gold_dim_geolocation_df.columns)
)

# Inspect schema
gold_dim_geolocation_df.printSchema()

# Preview dimension
display(
    gold_dim_geolocation_df.orderBy("geo_key").limit(15)
)


Gold Geolocation Dimension Created!
Source Geolocation Records: 738332
Unique ZIP Prefixes: 19015
Gold Geolocation Records: 19015
Gold Geolocation Columns: 10
root
 |-- geo_key: long (nullable = false)
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- geolocation_points: long (nullable = false)
 |-- distinct_cities: long (nullable = false)
 |-- distinct_states: long (nullable = false)
 |-- has_location_conflict: boolean (nullable = false)



geo_key,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_points,distinct_cities,distinct_states,has_location_conflict
1,01001,-23.550312214055925,-46.63403045780417,sao paulo,SP,11,2,1,true
2,01002,-23.54826028365169,-46.63535265262277,sao paulo,SP,6,2,1,true
3,01003,-23.548991491863838,-46.63565342364827,sao paulo,SP,11,2,1,true
4,01004,-23.54985392331218,-46.634820018830155,sao paulo,SP,14,2,1,true
5,01005,-23.549487008784876,-46.63665025950918,sao paulo,SP,13,2,1,true
6,01006,-23.550094744517867,-46.635951307336455,sao paulo,SP,7,2,1,true
7,01007,-23.550080154327947,-46.63707786026565,sao paulo,SP,16,2,1,true
8,01008,-23.545832387811174,-46.63577485680958,sao paulo,SP,13,2,1,true
9,01009,-23.54682324471825,-46.63646088115626,sao paulo,SP,11,2,1,true
10,01010,-23.546439302347167,-46.635260393394404,sao paulo,SP,6,2,1,true


In [0]:

# Build location combination audit

geo_location_audit_df = (
    geo_base_df
    .groupBy(
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state"
    )
    .agg(
        F.count("*").alias("source_occurrences"),

        F.avg("geolocation_lat").alias("average_latitude"),

        F.avg("geolocation_lng").alias("average_longitude")
    )
)

# Identify the selected representative location
representative_pairs_df = (
    gold_dim_geolocation_df
    .select(
        "geolocation_zip_code_prefix",
        F.col("geolocation_city").alias("selected_city"),
        F.col("geolocation_state").alias("selected_state")
    )
)

geo_location_audit_df = (
    geo_location_audit_df
    .join(
        representative_pairs_df,
        "geolocation_zip_code_prefix",
        "left"
    )
    .withColumn(
        "is_representative_location",
        (
            (F.col("geolocation_city") == F.col("selected_city")) &
            (F.col("geolocation_state") == F.col("selected_state"))
        )
    )
    .withColumn(
        "has_location_conflict",
        F.col("geolocation_zip_code_prefix").isin(
            [str(i) for i in []]
        )
    )
    .drop("selected_city", "selected_state")
)

print(
    "Location Combination Audit Records:",
    geo_location_audit_df.count()
)

display(
    geo_location_audit_df
    .filter(~F.col("is_representative_location"))
    .limit(20)
)


Location Combination Audit Records: 27911


geolocation_zip_code_prefix,geolocation_city,geolocation_state,source_occurrences,average_latitude,average_longitude,is_representative_location,has_location_conflict
26022,nova iguaçu,RJ,23,-22.72733352053513,-43.452647035374426,false,false
87053,maringá,PR,16,-23.461160418792414,-51.917312815746996,false,false
17520,marília,SP,7,-22.239148323969125,-49.92575314419428,false,false
04413,são paulo,SP,3,-23.67089888566338,-46.63987365775815,false,false
02142,são paulo,SP,1,-23.5100047784509,-46.5757172250109,false,false
74020,goiânia,GO,4,-16.673445310392992,-49.2561149640997,false,false
54727,são lourenço da mata,PE,1,-7.9885255632931,-35.04824557864501,false,false
14095,ribeirão preto,SP,16,-21.191619402987527,-47.76536024913286,false,false
17212,jaú,SP,8,-22.312683226570158,-48.54708201017002,false,false
70362,brasília,DF,1,-15.817063499999996,-47.90778075,false,false


In [0]:
geo_location_audit_df = (
    geo_location_audit_df
    .drop("has_location_conflict")
    .join(
        gold_dim_geolocation_df.select(
            "geolocation_zip_code_prefix",
            "has_location_conflict"
        ),
        "geolocation_zip_code_prefix",
        "left"
    )
)

In [0]:

# ==========================================
# GEOLOCATION DIMENSION VALIDATION
# ==========================================

# 1. Total records
total_geo = gold_dim_geolocation_df.count()

print("Gold Geolocation Records:", total_geo)


# 2. Unique ZIP prefixes
unique_zips = (
    gold_dim_geolocation_df
    .select("geolocation_zip_code_prefix")
    .distinct()
    .count()
)

print("Unique ZIP Prefixes:", unique_zips)


# 3. Unique geo keys
unique_geo_keys = (
    gold_dim_geolocation_df
    .select("geo_key")
    .distinct()
    .count()
)

print("Unique Geo Keys:", unique_geo_keys)


# 4. Duplicate ZIP prefixes
print(
    "Duplicate ZIP Prefixes:",
    total_geo - unique_zips
)


# 5. Column count
print(
    "Gold Geolocation Columns:",
    len(gold_dim_geolocation_df.columns)
)


# 6. ZIP-code length validation
print(
    "Invalid ZIP Lengths:",
    gold_dim_geolocation_df.filter(
        F.length("geolocation_zip_code_prefix") != 5
    ).count()
)


# 7. Coordinate validation
print(
    "Invalid Latitude:",
    gold_dim_geolocation_df.filter(
        ~F.col("geolocation_lat").between(-90, 90) |
        F.isnan("geolocation_lat")
    ).count()
)

print(
    "Invalid Longitude:",
    gold_dim_geolocation_df.filter(
        ~F.col("geolocation_lng").between(-180, 180) |
        F.isnan("geolocation_lng")
    ).count()
)


# 8. Location conflicts
print(
    "ZIPs With Multiple Cities:",
    gold_dim_geolocation_df.filter(
        F.col("distinct_cities") > 1
    ).count()
)

print(
    "ZIPs With Multiple States:",
    gold_dim_geolocation_df.filter(
        F.col("distinct_states") > 1
    ).count()
)

print(
    "ZIPs With Any Location Conflict:",
    gold_dim_geolocation_df.filter(
        F.col("has_location_conflict")
    ).count()
)


# 9. NULL validation
print("NULL VALUE ANALYSIS:")

display(
    gold_dim_geolocation_df.agg(
        *[
            F.count(
                F.when(F.col(c).isNull(), 1)
            ).alias(c)
            for c in gold_dim_geolocation_df.columns
        ]
    )
)


Gold Geolocation Records: 19015
Unique ZIP Prefixes: 19015
Unique Geo Keys: 19015
Duplicate ZIP Prefixes: 0
Gold Geolocation Columns: 10
Invalid ZIP Lengths: 0
Invalid Latitude: 0
Invalid Longitude: 0
ZIPs With Multiple Cities: 8555
ZIPs With Multiple States: 8
ZIPs With Any Location Conflict: 8558
NULL VALUE ANALYSIS:


geo_key,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_points,distinct_cities,distinct_states,has_location_conflict
0,0,0,0,0,0,0,0,0,0


In [0]:

from pyspark.sql import functions as F

# 1. Replace global row_number key with deterministic ZIP-based key

gold_dim_geolocation_df = (
    gold_dim_geolocation_df
    .drop("geo_key")
    .withColumn(
        "geo_key",
        F.col("geolocation_zip_code_prefix").cast("long")
    )
    .select(
        "geo_key",
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state",
        "geolocation_points",
        "distinct_cities",
        "distinct_states",
        "has_location_conflict"
    )
)

# 2. Rebuild the audit table with the correct conflict flag

geo_location_audit_df = (
    geo_base_df
    .groupBy(
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state"
    )
    .agg(
        F.count("*").alias("source_occurrences"),
        F.avg("geolocation_lat").alias("average_latitude"),
        F.avg("geolocation_lng").alias("average_longitude")
    )
    .join(
        gold_dim_geolocation_df.select(
            "geolocation_zip_code_prefix",
            F.col("geolocation_city").alias("selected_city"),
            F.col("geolocation_state").alias("selected_state"),
            "has_location_conflict"
        ),
        "geolocation_zip_code_prefix",
        "left"
    )
    .withColumn(
        "is_representative_location",
        (
            (F.col("geolocation_city") == F.col("selected_city")) &
            (F.col("geolocation_state") == F.col("selected_state"))
        )
    )
    .drop("selected_city", "selected_state")
)

# 3. Check key uniqueness

assert gold_dim_geolocation_df.count() == (
    gold_dim_geolocation_df
    .select("geo_key")
    .distinct()
    .count()
), "Duplicate geo_key values found!"

# 4. Save Gold dimension

(
    gold_dim_geolocation_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_geolocation")
)

# 5. Save location audit

(
    geo_location_audit_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.geolocation_location_audit")
)

print("Both Gold Geolocation tables saved successfully!")

print(
    "Dimension records:",
    spark.table("workspace.gold.dim_geolocation").count()
)

print(
    "Audit records:",
    spark.table("workspace.gold.geolocation_location_audit").count()
)


Both Gold Geolocation tables saved successfully!
Dimension records: 19015
Audit records: 27911


In [0]:
# Verify Gold and Silver source tables

tables_to_check = [
    "workspace.silver.silver_orders",
    "workspace.silver.silver_order_items",
    "workspace.gold.dim_customers",
    "workspace.gold.dim_products",
    "workspace.gold.dim_sellers",
    "workspace.gold.dim_date"
]

for table in tables_to_check:
    df = spark.table(table)
    print(f"{table}: {df.count()} records")
    df.printSchema()

workspace.silver.silver_orders: 99441 records
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

workspace.silver.silver_order_items: 112650 records
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

workspace.gold.dim_customers: 99441 records
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_

In [0]:

from pyspark.sql import functions as F

# Load source tables
orders_df = spark.table("workspace.silver.silver_orders")
order_items_df = spark.table("workspace.silver.silver_order_items")

# Join orders with their individual order items
fact_sales_base_df = (
    order_items_df.alias("oi")
    .join(
        orders_df.alias("o"),
        on="order_id",
        how="inner"
    )
    .select(
        "order_id",
        F.col("oi.order_item_id"),
        F.col("o.customer_id"),
        F.col("oi.product_id"),
        F.col("oi.seller_id"),
        F.col("o.order_status"),
        F.col("o.order_purchase_timestamp"),
        F.col("o.order_approved_at"),
        F.col("o.order_delivered_carrier_date"),
        F.col("o.order_delivered_customer_date"),
        F.col("o.order_estimated_delivery_date"),
        F.col("oi.shipping_limit_date"),
        F.col("oi.price"),
        F.col("oi.freight_value")
    )
)

# Display results
print("Fact Sales Base Records:", fact_sales_base_df.count())

print(
    "Unique Order Items:",
    fact_sales_base_df
    .select("order_id", "order_item_id")
    .distinct()
    .count()
)

print("Columns:", len(fact_sales_base_df.columns))

display(fact_sales_base_df.limit(10))


Fact Sales Base Records: 112650
Unique Order Items: 112650
Columns: 14


order_id,order_item_id,customer_id,product_id,seller_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,shipping_limit_date,price,freight_value
0005f50442cb953dcd1d21e1fb923495,1,351d3cb2cee3c7fd0af6616c82df21d3,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,delivered,2018-07-02T13:59:39.000Z,2018-07-02T14:10:56.000Z,2018-07-03T14:25:00.000Z,2018-07-04T17:28:31.000Z,2018-07-23T00:00:00.000Z,2018-07-06T14:10:56.000Z,53.99,11.4
00063b381e2406b52ad429470734ebd5,1,6a899e55865de6549a58d2c6845e5604,f177554ea93259a5b282f24e33f65ab6,8602a61d680a10a82cceeeda0d99ea3d,delivered,2018-07-27T17:21:27.000Z,2018-07-27T18:00:06.000Z,2018-07-30T14:52:00.000Z,2018-08-07T13:56:52.000Z,2018-08-07T00:00:00.000Z,2018-07-31T17:30:39.000Z,45.0,12.98
001021efaa8636c29475e7734483457d,1,2dfbf74859104caf100df3720a1d833d,5d7c23067ed3fc8c6e699b9373d5890b,6560211a19b47992c3666cc44a7e94c0,delivered,2018-02-27T09:27:14.000Z,2018-02-27T09:35:41.000Z,2018-02-27T21:43:46.000Z,2018-03-08T20:06:33.000Z,2018-03-23T00:00:00.000Z,2018-03-05T09:35:41.000Z,49.0,15.1
001dbc16dc51075e987543d23a0507c7,1,698a74f33469466fa4172e829505d1c6,777d2e438a1b645f3aec9bd57e92672c,4a3ca9315b744ce9f8e9374361493884,delivered,2017-01-28T13:17:57.000Z,2017-01-28T13:32:16.000Z,2017-02-01T15:59:46.000Z,2017-02-13T13:17:47.000Z,2017-03-20T00:00:00.000Z,2017-02-01T13:17:57.000Z,69.9,18.0
00276d5c3491fbf55305e26891040df9,1,54686c41e86854e35879efbd1c2cab1b,71c89e9d8fa274b017279aa322cd0e19,8f78f0903005064036736c7173a5c2ed,delivered,2018-02-13T11:47:21.000Z,2018-02-13T12:55:57.000Z,2018-02-20T23:19:10.000Z,2018-03-16T19:57:52.000Z,2018-03-22T00:00:00.000Z,2018-02-27T11:55:57.000Z,44.9,23.22
003423b755b562962a6225a8de40d12e,1,18f1bb6325d50619d5c13b7a25b869fd,84904413b82eee333a3f79137e8d197e,23c38debaffe4a25a30fdbd9b586a13f,delivered,2018-07-08T11:28:17.000Z,2018-07-10T04:25:09.000Z,2018-07-13T15:00:00.000Z,2018-07-24T21:33:38.000Z,2018-08-08T00:00:00.000Z,2018-07-18T04:25:09.000Z,232.75,28.58
003a94f778ef8cfd50247c8c1b582257,1,da108bc410acffb1400cf9caea65a332,452f66a0f164cac57802e2cea93188ac,f80edd2c5aaa505cc4b0a3b219abf4b8,delivered,2018-08-03T14:52:31.000Z,2018-08-03T15:05:14.000Z,2018-08-06T14:12:00.000Z,2018-08-24T00:18:38.000Z,2018-08-15T00:00:00.000Z,2018-08-07T15:05:14.000Z,39.9,18.08
004345d16a1ab2c21962992c721c8643,1,3b723b649feb48f3cfbb9c728efb3d7d,d0fb1e667e989933a80444f93da833c0,0be8ff43f22e456b4e0371b2245e4d01,delivered,2018-07-04T06:45:05.000Z,2018-07-05T16:09:18.000Z,null,2018-07-11T17:26:37.000Z,2018-07-30T00:00:00.000Z,2018-07-10T07:31:33.000Z,37.9,15.37
0046e1d57f4c07c8c92ab26be8c3dfc0,1,ae55e228ded796581eccc67045516023,ff6caf9340512b8bf6d2a2a6df032cfa,38e6dada03429a47197d5d584d793b41,delivered,2017-09-26T15:24:03.000Z,2017-09-26T15:49:17.000Z,2017-09-28T18:42:42.000Z,2017-09-29T18:29:50.000Z,2017-10-11T00:00:00.000Z,2017-10-02T15:49:17.000Z,7.79,7.78
004eab0fd8f28adaf8d488976f77febe,1,c476ddfbabfa4624603e0b7f8e245057,0c753fe6a58dfa7a27bda5de76e779c3,7586919161935337bf6b6d7ff5779648,delivered,2017-08-02T15:32:46.000Z,2017-08-02T15:45:17.000Z,2017-08-02T20:32:44.000Z,2017-08-03T18:17:47.000Z,2017-08-15T00:00:00.000Z,2017-08-08T15:45:17.000Z,21.5,11.98


In [0]:

# Load Gold dimensions
dim_customers_df = spark.table("workspace.gold.dim_customers")
dim_products_df = spark.table("workspace.gold.dim_products")
dim_sellers_df = spark.table("workspace.gold.dim_sellers")

# Join dimensions with Fact Sales base
fact_sales_enriched_df = (
    fact_sales_base_df.alias("f")

    .join(
        dim_customers_df.alias("c"),
        on="customer_id",
        how="left"
    )

    .join(
        dim_products_df.alias("p"),
        on="product_id",
        how="left"
    )

    .join(
        dim_sellers_df.alias("s"),
        on="seller_id",
        how="left"
    )

    .select(
        "order_id",
        "order_item_id",
        "customer_id",
        "customer_unique_id",
        "product_id",
        "product_category_name_english",
        "seller_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "shipping_limit_date",
        "price",
        "freight_value",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    )
)

print("Enriched Fact Sales Records:", fact_sales_enriched_df.count())

print(
    "Unique Order Items:",
    fact_sales_enriched_df
    .select("order_id", "order_item_id")
    .distinct()
    .count()
)

print("Columns:", len(fact_sales_enriched_df.columns))

display(fact_sales_enriched_df.limit(10))


Enriched Fact Sales Records: 112650
Unique Order Items: 112650
Columns: 22


order_id,order_item_id,customer_id,customer_unique_id,product_id,product_category_name_english,seller_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,shipping_limit_date,price,freight_value,customer_zip_code_prefix,customer_city,customer_state,seller_zip_code_prefix,seller_city,seller_state
000e63d38ae8c00bbcb5a30573b99628,1,98884e672c5ba85f4394f2044e1a3eab,860fc00d3154ce2346c43ebe47b9b6ce,553e0e7590d3116a072507a3635d2877,bed_bath_table,1c129092bf23f28a5930387c980c0dfc,delivered,2018-03-23T19:48:26.000Z,2018-03-23T20:07:49.000Z,2018-03-26T21:38:48.000Z,2018-03-27T14:51:47.000Z,2018-04-05T00:00:00.000Z,2018-03-29T20:07:49.000Z,47.9,8.88,04679,sao paulo,SP,02972,sao paulo,SP
00125cb692d04887809806618a2a145f,1,8afb90a97ee661103014329b1bcea1a2,7ac26eac431c6848694a2de6f0327524,1c0c0093a48f13ba70d0c6b0a9157cb7,furniture_decor,41b39e28db005d9731d9d485a83b4c38,delivered,2017-03-23T12:21:17.000Z,2017-03-23T13:05:42.000Z,2017-03-27T08:58:04.000Z,2017-04-07T15:32:47.000Z,2017-04-20T00:00:00.000Z,2017-03-29T13:05:42.000Z,109.9,25.51,75123,anapolis,GO,09220,santo andre,SP
0013503b13da1eac686219390b7d641b,1,f7277fce45fad3165379c15cacf5ddbe,1544f85dff3797491b60f2a5241d14c1,38afdf723b95d455b418a0f57d623c6b,furniture_decor,1554a68530182680ad5c8b042c3ab563,delivered,2017-12-05T23:40:59.000Z,2017-12-07T03:16:36.000Z,2017-12-07T22:59:30.000Z,2017-12-20T22:46:28.000Z,2017-12-28T00:00:00.000Z,2017-12-13T03:16:36.000Z,119.9,17.32,06160,osasco,SP,37580,monte siao,MG
0035c0b07126fe9c24a325216fb96064,1,c8c4991d84dbfef1f61c6152486db6d4,9392534851a5f1f12396b5f897f9f2ad,ec02a5d380128f7a188e9ce8f3ddd832,cool_stuff,8444e55c1f13cd5c179851e5ca5ebd00,delivered,2017-11-24T01:08:37.000Z,2017-11-24T01:29:45.000Z,2017-11-28T17:17:17.000Z,2017-12-07T16:59:21.000Z,2018-01-03T00:00:00.000Z,2017-12-12T01:29:45.000Z,131.9,18.54,13075,campinas,SP,37584,congonhal,MG
0035e6b7ade84b3f5b86bd49814793df,1,3a0cc7c9af6abf8868dad5a1de4a16e1,fdfa1cc68f121149d8fcc27ee96e134c,71a5f1c2a5fd9889ef26b5ac22aec9c6,furniture_decor,537eb890efff034a88679788b647c564,delivered,2018-02-21T02:46:23.000Z,2018-02-21T07:10:40.000Z,2018-02-22T00:51:33.000Z,2018-02-26T16:49:02.000Z,2018-03-13T00:00:00.000Z,2018-02-27T03:31:08.000Z,19.9,14.1,02019,sao paulo,SP,20270,rio de janeiro,RJ
003d9fc84ad902adf2265248b5ffe1a4,1,8c66482ec333eae596c93a9131981a9a,f26b89209f5378f95d8e664073b76419,3fbc0ef745950c7932d5f2a446189725,health_beauty,06a2c3af7b3aee5d69171b0e14f0ee87,delivered,2018-08-19T19:43:01.000Z,2018-08-20T19:35:22.000Z,2018-08-22T10:06:00.000Z,2018-08-27T18:17:49.000Z,2018-09-05T00:00:00.000Z,2018-08-22T19:35:22.000Z,66.99,27.75,05542,sao paulo,SP,65072,sao luis,MA
004eab0fd8f28adaf8d488976f77febe,1,c476ddfbabfa4624603e0b7f8e245057,5b861df2503dbc8ca307dfb7e18b300b,0c753fe6a58dfa7a27bda5de76e779c3,sports_leisure,7586919161935337bf6b6d7ff5779648,delivered,2017-08-02T15:32:46.000Z,2017-08-02T15:45:17.000Z,2017-08-02T20:32:44.000Z,2017-08-03T18:17:47.000Z,2017-08-15T00:00:00.000Z,2017-08-08T15:45:17.000Z,21.5,11.98,20720,rio de janeiro,RJ,21921,rio de janeiro,RJ
00995d799817ecc3bd2abd8fbe59c430,1,266636baec3cfa8d854713f0b465a2cb,f507945d757fe904ec16d8e7bb7ba7ad,f3b8bfa5b86249e75e5c0632acc0e82e,health_beauty,620c87c171fb2a6dd6e8bb4dec959fc6,delivered,2018-07-26T11:25:18.000Z,2018-07-26T11:45:13.000Z,2018-08-01T14:05:00.000Z,2018-08-02T18:56:52.000Z,2018-08-08T00:00:00.000Z,2018-08-01T11:45:13.000Z,49.9,12.92,24452,sao goncalo,RJ,25645,petropolis,RJ
00b402e8a76d3da83714c52d3977cc2d,1,ffca23943c4b620ea24812018373fa3f,e2421eb09087faf4f51661d176aee08d,72d6074ef5c81ec3102b3f2a052c5f49,health_beauty,edb1ef5e36e0c8cd84eb3c9b003e486d,delivered,2018-01-01T18:03:43.000Z,2018-01-01T18:11:33.000Z,2018-01-03T19:52:24.000Z,2018-01-19T12:08:06.000Z,2018-01-31T00:00:00.000Z,2018-01-05T18:11:33.000Z,139.65,26.26,45810,porto seguro,BA,25957,teresopolis,RJ
00bdcdda88e6b02977fc6ce3d412c600,1,45ba03e2c6bbb5dc48131ba32ec3ae5e,1

In [0]:

from pyspark.sql import functions as F

# Load dimensions
dim_date_df = spark.table("workspace.gold.dim_date")

dim_geolocation_df = spark.table(
    "workspace.gold.dim_geolocation"
)

# Prepare date dimension
date_lookup_df = dim_date_df.select(
    F.col("full_date").alias("purchase_date"),
    F.col("date_key").alias("purchase_date_key")
)

# Prepare customer geolocation lookup
customer_geo_df = dim_geolocation_df.select(
    F.col("geolocation_zip_code_prefix").alias("customer_geo_zip"),
    F.col("geo_key").alias("customer_geo_key")
)

# Prepare seller geolocation lookup
seller_geo_df = dim_geolocation_df.select(
    F.col("geolocation_zip_code_prefix").alias("seller_geo_zip"),
    F.col("geo_key").alias("seller_geo_key")
)

# Add date and geographic keys
fact_sales_final_df = (
    fact_sales_enriched_df

    # Purchase date key
    .withColumn(
        "purchase_date",
        F.to_date("order_purchase_timestamp")
    )

    .join(
        date_lookup_df,
        on="purchase_date",
        how="left"
    )

    # Customer geographic key
    .withColumn(
        "customer_geo_zip",
        F.lpad(
            F.trim(F.col("customer_zip_code_prefix")),
            5,
            "0"
        )
    )

    .join(
        customer_geo_df,
        on="customer_geo_zip",
        how="left"
    )

    # Seller geographic key
    .withColumn(
        "seller_geo_zip",
        F.lpad(
            F.trim(F.col("seller_zip_code_prefix")),
            5,
            "0"
        )
    )

    .join(
        seller_geo_df,
        on="seller_geo_zip",
        how="left"
    )

    .drop(
        "customer_geo_zip",
        "seller_geo_zip"
    )
)

print(
    "Fact Sales Records:",
    fact_sales_final_df.count()
)

print(
    "Unique Order Items:",
    fact_sales_final_df
    .select("order_id", "order_item_id")
    .distinct()
    .count()
)

print(
    "Columns:",
    len(fact_sales_final_df.columns)
)

# Check missing dimension keys
display(
    fact_sales_final_df.agg(
        F.count(
            F.when(F.col("purchase_date_key").isNull(), 1)
        ).alias("missing_purchase_date_key"),

        F.count(
            F.when(F.col("customer_geo_key").isNull(), 1)
        ).alias("missing_customer_geo_key"),

        F.count(
            F.when(F.col("seller_geo_key").isNull(), 1)
        ).alias("missing_seller_geo_key")
    )
)

display(fact_sales_final_df.limit(10))


Fact Sales Records: 112650
Unique Order Items: 112650
Columns: 26


missing_purchase_date_key,missing_customer_geo_key,missing_seller_geo_key
0,302,253


purchase_date,order_id,order_item_id,customer_id,customer_unique_id,product_id,product_category_name_english,seller_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,shipping_limit_date,price,freight_value,customer_zip_code_prefix,customer_city,customer_state,seller_zip_code_prefix,seller_city,seller_state,purchase_date_key,customer_geo_key,seller_geo_key
2018-04-25,00335f75ea6a4455b524a8dd177067f5,1,32b84239e3f3904ac162d832454770c7,a666bec8af192f2238a0b719a7063a00,e1bf2f209fd2b08e1af3ed71732d9008,pet_shop,70c27847eca8195c983ed7e798c56743,delivered,2018-04-25T21:47:26.000Z,2018-04-25T22:10:29.000Z,2018-04-27T14:49:00.000Z,2018-05-03T22:16:51.000Z,2018-05-28T00:00:00.000Z,2018-05-02T22:10:29.000Z,168.8,8.21,23093,rio de janeiro,RJ,20930,rio de janeiro,RJ,20180425,23093,20930
2018-02-02,00378c6c981f234634c0b9d6128df6dd,1,f3476df115c5d26bdb6bfb71b5fb66af,cd2fd2a38f5903eb2a7377b548e0bfdd,38fa750a3a3b3204f169c86a3284d387,sports_leisure,218d46b86c1881d022bce9c68a7d4b15,delivered,2018-02-02T19:39:46.000Z,2018-02-02T19:50:27.000Z,2018-02-05T18:06:49.000Z,2018-02-26T21:36:00.000Z,2018-02-26T00:00:00.000Z,2018-02-08T19:50:27.000Z,41.0,11.85,04826,sao paulo,SP,14070,ribeirao preto,SP,20180202,4826,14070
2017-12-25,004ba47bb589cdbc485e6025bd6d3c81,1,ab30bf72cae845f3814c7baa50ca549a,8b9c297f4874622ea42c249bba3854d7,8c5876b1c7768217964f353bc7e64393,telephony,0db783cfcd3b73998abc6e10e59a102f,delivered,2017-12-25T20:40:15.000Z,2017-12-25T20:48:10.000Z,2017-12-26T20:39:05.000Z,2017-12-29T20:24:23.000Z,2018-01-19T00:00:00.000Z,2017-12-29T20:48:10.000Z,49.0,11.85,15775,santa fe do sul,SP,11010,santos,SP,20171225,15775,11010
2017-11-24,007ec946d4f798caf2b43c9d44c56421,1,3ace63287bc8b41172016c06a5093808,25e6bea0f352b9c61d1bed324da0f5cb,779559842fc122d55edbd03153f35e81,sports_leisure,f8db351d8c4c4c22c6835c19a46f01b0,delivered,2017-11-24T13:05:15.000Z,2017-11-25T04:15:38.000Z,2017-11-28T19:27:06.000Z,2017-12-06T22:38:40.000Z,2017-12-26T00:00:00.000Z,2017-11-30T04:15:38.000Z,27.9,15.1,95670,gramado,RS,13324,salto,SP,20171124,95670,13324
2018-05-11,008990da719764fb392914d8ab53ad04,1,cc4e2f483459498c9ce82a70ff415752,d3391cadc57741a0defe8a569e476813,a8115d34a4b7b74388da609692162c37,computers_accessories,85d9eb9ddc5d00ca9336a2219c97bb13,delivered,2018-05-11T19:20:54.000Z,2018-05-11T19:37:41.000Z,2018-05-24T13:47:00.000Z,2018-06-05T14:18:36.000Z,2018-06-19T00:00:00.000Z,2018-06-01T19:37:41.000Z,35.9,18.23,27930,macae,RJ,31255,belo horizonte,MG,20180511,27930,31255
2017-12-09,00946f674d880be1f188abc10ad7cf46,1,9583b047c90d4aa1c5c68b4065ba7f4e,1d74273c053909be057f50f65b48522f,4dcb49b9ca7e48d2f108d40caa77caa2,pet_shop,5cf13accae3222c70a9cac40818ae839,delivered,2017-12-09T19:11:22.000Z,2017-12-09T19:30:37.000Z,2017-12-11T18:47:04.000Z,2017-12-17T10:19:09.000Z,2018-01-05T00:00:00.000Z,2017-12-14T19:30:37.000Z,99.9,18.57,06114,osasco,SP,38700,patos de minas,MG,20171209,6114,38700
2017-12-09,00946f674d880be1f188abc10ad7cf46,2,9583b047c90d4aa1c5c68b4065ba7f4e,1d74273c053909be057f50f65b48522f,9bb2d066e4b33b624cbdfec7d50b3dcb,pet_shop,5cf13accae3222c70a9cac40818ae839,delivered,2017-12-09T19:11:22.000Z,2017-12-09T19:30:37.000Z,2017-12-11T18:47:04.000Z,2017-12-17T10:19:09.000Z,2018-01-05T00:00:00.000Z,2017-12-14T19:30:37.000Z,99.9,14.35,06114,osasco,SP,38700,patos de minas,MG,20171209,6114,38700
2018-07-25,00a1e977cf6e672402963940db55d919,1,e0dc8ea464de23db3b4aa5fa6ae07cc3,383fa1a759c0437605732a46d4f43d2c,4834f24983a4f056e5ae64265f723aaf,electronics,5656537e588803a555b8eb41f07a944b,delivered,2018-07-25T16:25:31.000Z,2018-07-25T16:35:16.000Z,2018-07-26T11:53:00.000Z,2018-07-27T16:12:04.000Z,2018-08-02T00:00:00.000Z,2018-07-27T16:35:16.000Z,24.89,7.75,71918,brasilia,DF,72015,brasilia,DF,20180725,71918,72015
2017-08-02,00b676b01c289cc661c6f7732492771a,1,ca9a6ae226341827c9614ce7568db46c,15b521471c36ed411359347ff8257b79,121b9686b9929855d823981fc655a6fe,housewares,c4fb51fb1c5b

In [0]:
from pyspark.sql import functions as F

# Missing dimension key analysis
display(
    fact_sales_final_df.agg(
        F.count(
            F.when(F.col("purchase_date_key").isNull(), 1)
        ).alias("missing_purchase_date_key"),

        F.count(
            F.when(F.col("customer_geo_key").isNull(), 1)
        ).alias("missing_customer_geo_key"),

        F.count(
            F.when(F.col("seller_geo_key").isNull(), 1)
        ).alias("missing_seller_geo_key"),

        F.count(
            F.when(F.col("customer_unique_id").isNull(), 1)
        ).alias("missing_customer_dimension"),

        F.count(
            F.when(F.col("product_category_name_english").isNull(), 1)
        ).alias("missing_product_category_translation")
    )
)

missing_purchase_date_key,missing_customer_geo_key,missing_seller_geo_key,missing_customer_dimension,missing_product_category_translation
0,302,253,0,1627


In [0]:

from pyspark.sql import functions as F

# Add sales measures
fact_sales_final_df = (
    fact_sales_final_df

    .withColumn(
        "total_item_value",
        F.round(
            F.coalesce(F.col("price"), F.lit(0.0)) +
            F.coalesce(F.col("freight_value"), F.lit(0.0)),
            2
        )
    )

    .withColumn(
        "order_item_count",
        F.lit(1)
    )
)

# Validate sales measures
display(
    fact_sales_final_df.agg(
        F.count("*").alias("total_records"),

        F.sum("order_item_count").alias("total_order_items"),

        F.round(F.sum("price"), 2).alias("total_product_sales"),

        F.round(F.sum("freight_value"), 2).alias("total_freight_value"),

        F.round(F.sum("total_item_value"), 2).alias("total_sales_including_freight"),

        F.min("total_item_value").alias("minimum_item_value"),

        F.max("total_item_value").alias("maximum_item_value"),

        F.count(
            F.when(F.col("total_item_value") < 0, 1)
        ).alias("negative_item_values")
    )
)


total_records,total_order_items,total_product_sales,total_freight_value,total_sales_including_freight,minimum_item_value,maximum_item_value,negative_item_values
112650,112650,1.35916437E7,2251909.54,1.584355324E7,6.08,6929.31,0


In [0]:

from pyspark.sql import functions as F

# Final Fact Sales validation

total_records = fact_sales_final_df.count()

unique_order_items = (
    fact_sales_final_df
    .select("order_id", "order_item_id")
    .distinct()
    .count()
)

duplicate_records = total_records - unique_order_items

null_order_ids = fact_sales_final_df.filter(
    F.col("order_id").isNull()
).count()

null_item_ids = fact_sales_final_df.filter(
    F.col("order_item_id").isNull()
).count()

null_product_prices = fact_sales_final_df.filter(
    F.col("price").isNull()
).count()

null_freight_values = fact_sales_final_df.filter(
    F.col("freight_value").isNull()
).count()

print("========== FACT SALES FINAL VALIDATION ==========")

print("Total Records:", total_records)
print("Unique Order Items:", unique_order_items)
print("Duplicate Records:", duplicate_records)
print("NULL Order IDs:", null_order_ids)
print("NULL Order Item IDs:", null_item_ids)
print("NULL Product Prices:", null_product_prices)
print("NULL Freight Values:", null_freight_values)
print("Total Columns:", len(fact_sales_final_df.columns))

assert total_records == 112650
assert duplicate_records == 0
assert null_order_ids == 0
assert null_item_ids == 0

print("===============================================")
print("FACT SALES VALIDATION COMPLETED")


========== FACT SALES FINAL VALIDATION ==========
Total Records: 112650
Unique Order Items: 112650
Duplicate Records: 0
NULL Order IDs: 0
NULL Order Item IDs: 0
NULL Product Prices: 0
NULL Freight Values: 0
Total Columns: 28
FACT SALES VALIDATION COMPLETED


In [0]:

# Save Fact Sales Gold Delta table

(
    fact_sales_final_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.fact_sales")
)

# Verify saved table
saved_fact_sales_df = spark.table("workspace.gold.fact_sales")

print("FACT SALES SAVED SUCCESSFULLY!")
print("Table: workspace.gold.fact_sales")
print("Records:", saved_fact_sales_df.count())
print("Columns:", len(saved_fact_sales_df.columns))

saved_fact_sales_df.printSchema()


FACT SALES SAVED SUCCESSFULLY!
Table: workspace.gold.fact_sales
Records: 112650
Columns: 28
root
 |-- purchase_date: date (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)

In [0]:
# Load Silver Payments
silver_payments_df = spark.table(
    "workspace.silver.silver_payments"
)

print("Silver Payments Records:", silver_payments_df.count())

print("Columns:", len(silver_payments_df.columns))

silver_payments_df.printSchema()

display(silver_payments_df.limit(10))

Silver Payments Records: 103886
Columns: 5
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



order_id,payment_sequential,payment_type,payment_installments,payment_value
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12
771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16
3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84
1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09
0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95


In [0]:

from pyspark.sql import functions as F

# Load Silver Orders
silver_orders_df = spark.table(
    "workspace.silver.silver_orders"
)

# Join payments with orders
fact_payments_base_df = (
    silver_payments_df.alias("p")
    .join(
        silver_orders_df.alias("o"),
        on="order_id",
        how="left"
    )
    .select(
        F.col("p.order_id"),
        F.col("p.payment_sequential"),
        F.col("o.customer_id"),
        F.col("o.order_status"),
        F.col("o.order_purchase_timestamp"),
        F.col("p.payment_type"),
        F.col("p.payment_installments"),
        F.col("p.payment_value")
    )
)

# Validate record preservation
print(
    "Fact Payments Base Records:",
    fact_payments_base_df.count()
)

print(
    "Unique Payment Transactions:",
    fact_payments_base_df
    .select("order_id", "payment_sequential")
    .distinct()
    .count()
)

print(
    "Columns:",
    len(fact_payments_base_df.columns)
)

display(fact_payments_base_df.limit(10))


Fact Payments Base Records: 103886
Unique Payment Transactions: 103886
Columns: 8


order_id,payment_sequential,customer_id,order_status,order_purchase_timestamp,payment_type,payment_installments,payment_value
298fcdf1f73eb413e4d26d01b25bc1cd,1,a24e6f72471e9dbafcb292bc318f4859,delivered,2018-05-07T13:20:41.000Z,credit_card,2,96.12
b81ef226f3fe1789b1e8b2acac839d17,1,0a8556ac6be836b46b3e89920d59291c,delivered,2018-04-25T22:01:49.000Z,credit_card,8,99.33
ba78997921bbcdc1373bb41e913ab953,1,7a5d8efaaa1081f800628c30d2b0728f,delivered,2017-12-06T12:04:06.000Z,credit_card,8,107.78
0573b5e23cbd798006520e1d5b4c6714,1,c4e17eff78176dfe3401f03db0346f85,delivered,2017-07-04T20:32:10.000Z,boleto,1,51.95
3d7239c394a212faae122962df514ac7,1,4d7a9b8bba459dce09d1b8fa637a0cba,delivered,2017-06-05T10:45:54.000Z,credit_card,3,51.84
25e8ea4e93396b6fa0d3dd708e76c1bd,1,25b14b69de0b6e184ae6fe2755e478f9,delivered,2017-12-12T11:19:55.000Z,credit_card,1,65.71
771ee386b001f06208a7419e4fc1bbd7,1,206c0fcc717333d169ff95933fb47341,delivered,2017-06-23T13:11:27.000Z,credit_card,1,81.16
1f78449c87a54faf9e96e88ba1491fa9,1,68c9dafdf193586c290b19c71541357a,delivered,2018-07-22T14:07:35.000Z,credit_card,6,341.09
a9810da82917af2d9aefd1278f1dcfa0,1,f2c7fc58a9de810828715166c672f10a,delivered,2018-06-26T11:01:38.000Z,credit_card,1,24.39
42fdf880ba16b47b59251dd489d4441a,1,15fd6fb8f8312dbb4674e4518d6fa3b3,delivered,2018-05-21T13:59:17.000Z,credit_card,2,128.45


In [0]:

# Load Gold dimensions
dim_customers_df = spark.table("workspace.gold.dim_customers")
dim_date_df = spark.table("workspace.gold.dim_date")
dim_geolocation_df = spark.table("workspace.gold.dim_geolocation")

# Prepare date lookup
payment_date_lookup_df = dim_date_df.select(
    F.col("full_date").alias("purchase_date"),
    F.col("date_key").alias("purchase_date_key")
)

# Prepare customer geolocation lookup
payment_geo_lookup_df = dim_geolocation_df.select(
    F.col("geolocation_zip_code_prefix").alias("customer_geo_zip"),
    F.col("geo_key").alias("customer_geo_key")
)

# Enrich payment data
fact_payments_enriched_df = (
    fact_payments_base_df.alias("p")

    .join(
        dim_customers_df.alias("c"),
        on="customer_id",
        how="left"
    )

    .withColumn(
        "purchase_date",
        F.to_date("order_purchase_timestamp")
    )

    .join(
        payment_date_lookup_df,
        on="purchase_date",
        how="left"
    )

    .withColumn(
        "customer_geo_zip",
        F.lpad(
            F.trim(F.col("customer_zip_code_prefix")),
            5,
            "0"
        )
    )

    .join(
        payment_geo_lookup_df,
        on="customer_geo_zip",
        how="left"
    )

    .drop("customer_geo_zip")

    .select(
        "order_id",
        "payment_sequential",
        "customer_id",
        "customer_unique_id",
        "order_status",
        "order_purchase_timestamp",
        "purchase_date",
        "purchase_date_key",
        "customer_geo_key",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
        "payment_type",
        "payment_installments",
        "payment_value"
    )
)

print(
    "Enriched Payment Records:",
    fact_payments_enriched_df.count()
)

print(
    "Unique Payment Transactions:",
    fact_payments_enriched_df
    .select("order_id", "payment_sequential")
    .distinct()
    .count()
)

print(
    "Columns:",
    len(fact_payments_enriched_df.columns)
)

display(fact_payments_enriched_df.limit(10))


Enriched Payment Records: 103886
Unique Payment Transactions: 103886
Columns: 15


order_id,payment_sequential,customer_id,customer_unique_id,order_status,order_purchase_timestamp,purchase_date,purchase_date_key,customer_geo_key,customer_zip_code_prefix,customer_city,customer_state,payment_type,payment_installments,payment_value
1f78449c87a54faf9e96e88ba1491fa9,1,68c9dafdf193586c290b19c71541357a,2ecddf370925aff428e58b4f6021fe38,delivered,2018-07-22T14:07:35.000Z,2018-07-22,20180722,58410,58410,campina grande,PB,credit_card,6,341.09
b81ef226f3fe1789b1e8b2acac839d17,1,0a8556ac6be836b46b3e89920d59291c,708ab75d2a007f0564aedd11139c7708,delivered,2018-04-25T22:01:49.000Z,2018-04-25,20180425,39801,39801,teofilo otoni,MG,credit_card,8,99.33
0573b5e23cbd798006520e1d5b4c6714,1,c4e17eff78176dfe3401f03db0346f85,511c345b0383af6ef25406c564e4d6b6,delivered,2017-07-04T20:32:10.000Z,2017-07-04,20170704,17780,17780,lucelia,SP,boleto,1,51.95
25e8ea4e93396b6fa0d3dd708e76c1bd,1,25b14b69de0b6e184ae6fe2755e478f9,6f70c0b2f7552832ba46eb57b1c5651e,delivered,2017-12-12T11:19:55.000Z,2017-12-12,20171212,2652,02652,sao paulo,SP,credit_card,1,65.71
42fdf880ba16b47b59251dd489d4441a,1,15fd6fb8f8312dbb4674e4518d6fa3b3,4291db0da71914754618cd789aebcd56,delivered,2018-05-21T13:59:17.000Z,2018-05-21,20180521,18570,18570,conchas,SP,credit_card,2,128.45
771ee386b001f06208a7419e4fc1bbd7,1,206c0fcc717333d169ff95933fb47341,9a3526eee723accd014ec683afe5bf20,delivered,2017-06-23T13:11:27.000Z,2017-06-23,20170623,31230,31230,belo horizonte,MG,credit_card,1,81.16
298fcdf1f73eb413e4d26d01b25bc1cd,1,a24e6f72471e9dbafcb292bc318f4859,6e3c218d5f0434ddc4af3d6a60767bbf,delivered,2018-05-07T13:20:41.000Z,2018-05-07,20180507,13614,13614,leme,SP,credit_card,2,96.12
3d7239c394a212faae122962df514ac7,1,4d7a9b8bba459dce09d1b8fa637a0cba,9db67268a3fee1d4b13faaeb90af07c0,delivered,2017-06-05T10:45:54.000Z,2017-06-05,20170605,15200,15200,jose bonifacio,SP,credit_card,3,51.84
a9810da82917af2d9aefd1278f1dcfa0,1,f2c7fc58a9de810828715166c672f10a,a8b9d3a27068454b1c98cc67d4e31e6f,delivered,2018-06-26T11:01:38.000Z,2018-06-26,20180626,2422,02422,sao paulo,SP,credit_card,1,24.39
ba78997921bbcdc1373bb41e913ab953,1,7a5d8efaaa1081f800628c30d2b0728f,87695ed086ebd36f20404c82d20fca87,delivered,2017-12-06T12:04:06.000Z,2017-12-06,20171206,36060,36060,juiz de fora,MG,credit_card,8,107.78


In [0]:

from pyspark.sql import functions as F

display(
    fact_payments_enriched_df.agg(
        F.count(
            F.when(F.col("purchase_date_key").isNull(), 1)
        ).alias("missing_purchase_date_key"),

        F.count(
            F.when(F.col("customer_geo_key").isNull(), 1)
        ).alias("missing_customer_geo_key"),

        F.count(
            F.when(F.col("customer_unique_id").isNull(), 1)
        ).alias("missing_customer_dimension"),

        F.count(
            F.when(F.col("customer_id").isNull(), 1)
        ).alias("missing_customer_id"),

        F.count(
            F.when(F.col("payment_type").isNull(), 1)
        ).alias("missing_payment_type"),

        F.count(
            F.when(F.col("payment_value").isNull(), 1)
        ).alias("missing_payment_value")
    )
)


missing_purchase_date_key,missing_customer_geo_key,missing_customer_dimension,missing_customer_id,missing_payment_type,missing_payment_value
0,287,0,0,0,0


In [0]:

from pyspark.sql import functions as F

# Add payment measures
fact_payments_final_df = (
    fact_payments_enriched_df

    .withColumn(
        "payment_count",
        F.lit(1)
    )

    .withColumn(
        "is_zero_payment",
        F.col("payment_value") == 0
    )
)

# Calculate payment metrics
display(
    fact_payments_final_df.agg(
        F.count("*").alias("total_payment_records"),

        F.sum("payment_count").alias("total_payment_count"),

        F.round(
            F.sum("payment_value"), 2
        ).alias("total_payment_value"),

        F.round(
            F.avg("payment_value"), 2
        ).alias("average_payment_value"),

        F.min("payment_value").alias("minimum_payment_value"),

        F.max("payment_value").alias("maximum_payment_value"),

        F.sum(
            F.when(F.col("is_zero_payment"), 1).otherwise(0)
        ).alias("zero_payment_records"),

        F.count(
            F.when(F.col("payment_value") < 0, 1)
        ).alias("negative_payment_records")
    )
)


total_payment_records,total_payment_count,total_payment_value,average_payment_value,minimum_payment_value,maximum_payment_value,zero_payment_records,negative_payment_records
103886,103886,1.600887212E7,154.1,0.0,13664.08,9,0


In [0]:

from pyspark.sql import functions as F

total_records = fact_payments_final_df.count()

unique_transactions = (
    fact_payments_final_df
    .select("order_id", "payment_sequential")
    .distinct()
    .count()
)

duplicate_records = total_records - unique_transactions

print("========== FACT PAYMENTS VALIDATION ==========")

print("Total Records:", total_records)
print("Unique Payment Transactions:", unique_transactions)
print("Duplicate Records:", duplicate_records)

print(
    "NULL Order IDs:",
    fact_payments_final_df.filter(
        F.col("order_id").isNull()
    ).count()
)

print(
    "NULL Payment Sequential:",
    fact_payments_final_df.filter(
        F.col("payment_sequential").isNull()
    ).count()
)

print(
    "NULL Payment Values:",
    fact_payments_final_df.filter(
        F.col("payment_value").isNull()
    ).count()
)

print("Total Columns:", len(fact_payments_final_df.columns))

assert total_records == 103886
assert duplicate_records == 0

print("==============================================")
print("FACT PAYMENTS VALIDATION COMPLETED")


========== FACT PAYMENTS VALIDATION ==========
Total Records: 103886
Unique Payment Transactions: 103886
Duplicate Records: 0
NULL Order IDs: 0
NULL Payment Sequential: 0
NULL Payment Values: 0
Total Columns: 17
FACT PAYMENTS VALIDATION COMPLETED


In [0]:
(
    fact_payments_final_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.fact_payments")
)

# Verify the saved table
saved_fact_payments_df = spark.table("workspace.gold.fact_payments")

print("FACT PAYMENTS SAVED SUCCESSFULLY!")
print("Table: workspace.gold.fact_payments")
print("Records:", saved_fact_payments_df.count())
print("Columns:", len(saved_fact_payments_df.columns))

saved_fact_payments_df.printSchema()

FACT PAYMENTS SAVED SUCCESSFULLY!
Table: workspace.gold.fact_payments
Records: 103886
Columns: 17
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- purchase_date: date (nullable = true)
 |-- purchase_date_key: integer (nullable = true)
 |-- customer_geo_key: long (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- payment_count: integer (nullable = true)
 |-- is_zero_payment: boolean (nullable = true)



In [0]:
silver_reviews_df = spark.table("workspace.silver.silver_reviews")

print("Total Silver Reviews:", silver_reviews_df.count())
print("Total Columns:", len(silver_reviews_df.columns))

silver_reviews_df.printSchema()
display(silver_reviews_df.limit(5))

Total Silver Reviews: 99224
Total Columns: 7
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18T00:00:00.000Z,2018-01-18T21:46:59.000Z
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10T00:00:00.000Z,2018-03-11T03:05:13.000Z
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17T00:00:00.000Z,2018-02-18T14:36:24.000Z
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21T00:00:00.000Z,2017-04-21T22:02:06.000Z
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01T00:00:00.000Z,2018-03-02T10:26:53.000Z


In [0]:
from pyspark.sql import functions as F

silver_orders_df = spark.table("workspace.silver.silver_orders")

fact_reviews_base_df = (
    silver_reviews_df.alias("r")
    .join(
        silver_orders_df.alias("o"),
        on="order_id",
        how="left"
    )
    .select(
        F.col("r.review_id"),
        F.col("r.order_id"),
        F.col("o.customer_id"),
        F.col("o.order_status"),
        F.col("o.order_purchase_timestamp"),
        F.col("r.review_score"),
        F.col("r.review_comment_title"),
        F.col("r.review_comment_message"),
        F.col("r.review_creation_date"),
        F.col("r.review_answer_timestamp")
    )
)

print("Fact Reviews Base Records:", fact_reviews_base_df.count())

print(
    "Unique Review-Order Combinations:",
    fact_reviews_base_df
    .select("review_id", "order_id")
    .distinct()
    .count()
)

print("Total Columns:", len(fact_reviews_base_df.columns))

display(fact_reviews_base_df.limit(5))

Fact Reviews Base Records: 99224
Unique Review-Order Combinations: 99224
Total Columns: 10


review_id,order_id,customer_id,order_status,order_purchase_timestamp,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,de6dff97e5f1ba84a3cd9a3bc97df5f6,delivered,2017-04-09T17:41:13.000Z,5,null,Recebi bem antes do prazo estipulado.,2017-04-21T00:00:00.000Z,2017-04-21T22:02:06.000Z
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,41dcb106f807e993532d446263290104,delivered,2018-01-11T15:30:49.000Z,4,null,null,2018-01-18T00:00:00.000Z,2018-01-18T21:46:59.000Z
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5986b333ca0d44534a156a52a8e33a83,delivered,2018-02-10T10:59:03.000Z,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01T00:00:00.000Z,2018-03-02T10:26:53.000Z
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,e226dfed6544df5b7b87a48208690feb,delivered,2018-02-03T09:56:22.000Z,5,null,null,2018-02-17T00:00:00.000Z,2018-02-18T14:36:24.000Z
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,8a2e7ef9053dea531e4dc76bd6d853e6,delivered,2018-02-28T12:25:19.000Z,5,null,null,2018-03-10T00:00:00.000Z,2018-03-11T03:05:13.000Z


In [0]:
# Load Gold dimension tables

dim_customers_df = spark.table("workspace.gold.dim_customers")

dim_date_df = spark.table("workspace.gold.dim_date")

dim_geo_df = spark.table("workspace.gold.dim_geolocation")

print("All Gold dimension DataFrames loaded successfully!")

print("Customers:", dim_customers_df.count())
print("Date:", dim_date_df.count())
print("Geolocation:", dim_geo_df.count())

All Gold dimension DataFrames loaded successfully!
Customers: 99441
Date: 802
Geolocation: 19015


In [0]:
fact_reviews_enriched_df = (
    fact_reviews_base_df.alias("r")

    # Customer dimension
    .join(
        dim_customers_df.alias("c"),
        on="customer_id",
        how="left"
    )

    # Review creation date dimension
    .join(
        dim_date_df.alias("rd"),
        F.to_date(F.col("r.review_creation_date")) == F.col("rd.full_date"),
        how="left"
    )

    # Purchase date dimension
    .join(
        dim_date_df.alias("pd"),
        F.to_date(F.col("r.order_purchase_timestamp")) == F.col("pd.full_date"),
        how="left"
    )

    # Customer geolocation dimension
    .join(
        dim_geo_df.alias("g"),
        F.col("c.customer_zip_code_prefix") ==
        F.col("g.geolocation_zip_code_prefix"),
        how="left"
    )

    .select(
        F.col("r.*"),
        F.col("c.customer_unique_id"),
        F.col("c.customer_zip_code_prefix"),
        F.col("c.customer_city"),
        F.col("c.customer_state"),
        F.to_date(F.col("r.review_creation_date")).alias("review_date"),
        F.col("rd.date_key").alias("review_date_key"),
        F.to_date(F.col("r.order_purchase_timestamp")).alias("purchase_date"),
        F.col("pd.date_key").alias("purchase_date_key"),
        F.col("g.geo_key").alias("customer_geo_key")
    )
)

print("Fact Reviews Enriched Records:", fact_reviews_enriched_df.count())

print(
    "Unique Review-Order Combinations:",
    fact_reviews_enriched_df
    .select("review_id", "order_id")
    .distinct()
    .count()
)

print("Total Columns:", len(fact_reviews_enriched_df.columns))

display(fact_reviews_enriched_df.limit(5))

Fact Reviews Enriched Records: 99224
Unique Review-Order Combinations: 99224
Total Columns: 19


customer_id,review_id,order_id,order_status,order_purchase_timestamp,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,review_date,review_date_key,purchase_date,purchase_date_key,customer_geo_key
e226dfed6544df5b7b87a48208690feb,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,delivered,2018-02-03T09:56:22.000Z,5,null,null,2018-02-17T00:00:00.000Z,2018-02-18T14:36:24.000Z,1d47144362c14e94ccdd213e8ec277d5,44571,santo antonio de jesus,BA,2018-02-17,20180217,2018-02-03,20180203,44571
8a2e7ef9053dea531e4dc76bd6d853e6,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,delivered,2018-02-28T12:25:19.000Z,5,null,null,2018-03-10T00:00:00.000Z,2018-03-11T03:05:13.000Z,64190b91b656ab8f37eb89b93dc84584,13380,nova odessa,SP,2018-03-10,20180310,2018-02-28,20180228,13380
41dcb106f807e993532d446263290104,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,delivered,2018-01-11T15:30:49.000Z,4,null,null,2018-01-18T00:00:00.000Z,2018-01-18T21:46:59.000Z,68a5590b9926689be4e10f4ae2db21a8,06030,osasco,SP,2018-01-18,20180118,2018-01-11,20180111,6030
5986b333ca0d44534a156a52a8e33a83,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,delivered,2018-02-10T10:59:03.000Z,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01T00:00:00.000Z,2018-03-02T10:26:53.000Z,d16000272660a1fef81482ad75ba572a,89520,curitibanos,SC,2018-03-01,20180301,2018-02-10,20180210,89520
de6dff97e5f1ba84a3cd9a3bc97df5f6,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,delivered,2017-04-09T17:41:13.000Z,5,null,Recebi bem antes do prazo estipulado.,2017-04-21T00:00:00.000Z,2017-04-21T22:02:06.000Z,c8cf6cb6b838dc7a33ed199b825e8616,88735,gravatal,SC,2017-04-21,20170421,2017-04-09,20170409,88735


In [0]:
review_quality_checks = fact_reviews_enriched_df.select(
    F.count("*").alias("total_records"),

    F.sum(F.col("customer_id").isNull().cast("int"))
        .alias("missing_customer_id"),

    F.sum(F.col("customer_unique_id").isNull().cast("int"))
        .alias("missing_customer_dimension"),

    F.sum(F.col("review_date_key").isNull().cast("int"))
        .alias("missing_review_date_key"),

    F.sum(F.col("purchase_date_key").isNull().cast("int"))
        .alias("missing_purchase_date_key"),

    F.sum(F.col("customer_geo_key").isNull().cast("int"))
        .alias("missing_customer_geo_key"),

    F.sum(F.col("review_score").isNull().cast("int"))
        .alias("missing_review_score"),

    F.sum(
        (~F.col("review_score").between(1, 5)).cast("int")
    ).alias("invalid_review_score")
)

display(review_quality_checks)

total_records,missing_customer_id,missing_customer_dimension,missing_review_date_key,missing_purchase_date_key,missing_customer_geo_key,missing_review_score,invalid_review_score
99224,0,0,0,0,279,0,0


In [0]:
fact_reviews_final_df = (
    fact_reviews_enriched_df
    .withColumn("review_count", F.lit(1))
    .withColumn(
        "is_positive_review",
        F.col("review_score") >= 4
    )
)

display(
    fact_reviews_final_df
    .groupBy("review_score")
    .agg(
        F.count("*").alias("review_records"),
        F.sum("review_count").alias("total_review_count")
    )
    .orderBy("review_score")
)

review_score,review_records,total_review_count
1,11424,11424
2,3151,3151
3,8179,8179
4,19142,19142
5,57328,57328


In [0]:
fact_reviews_final_df.agg(
    F.count("*").alias("total_reviews"),

    F.sum("review_count").alias("total_review_count"),

    F.round(F.avg("review_score"), 2)
        .alias("average_review_score"),

    F.min("review_score").alias("minimum_review_score"),

    F.max("review_score").alias("maximum_review_score"),

    F.sum(
        F.col("is_positive_review").cast("int")
    ).alias("positive_reviews"),

    F.sum(
        (F.col("review_score") <= 2).cast("int")
    ).alias("negative_reviews"),

    F.sum(
        (F.col("review_score") == 3).cast("int")
    ).alias("neutral_reviews")
).show()

+-------------+------------------+--------------------+--------------------+--------------------+----------------+----------------+---------------+
|total_reviews|total_review_count|average_review_score|minimum_review_score|maximum_review_score|positive_reviews|negative_reviews|neutral_reviews|
+-------------+------------------+--------------------+--------------------+--------------------+----------------+----------------+---------------+
|        99224|             99224|                4.09|                   1|                   5|           76470|           14575|           8179|
+-------------+------------------+--------------------+--------------------+--------------------+----------------+----------------+---------------+



In [0]:
total_records = fact_reviews_final_df.count()

unique_reviews = (
    fact_reviews_final_df
    .select("review_id", "order_id")
    .distinct()
    .count()
)

duplicate_records = total_records - unique_reviews

print("========== FACT REVIEWS VALIDATION ==========")
print("Total Records:", total_records)
print("Unique Review-Order Combinations:", unique_reviews)
print("Duplicate Records:", duplicate_records)

print(
    "NULL Review IDs:",
    fact_reviews_final_df.filter(
        F.col("review_id").isNull()
    ).count()
)

print(
    "NULL Order IDs:",
    fact_reviews_final_df.filter(
        F.col("order_id").isNull()
    ).count()
)

print(
    "NULL Review Scores:",
    fact_reviews_final_df.filter(
        F.col("review_score").isNull()
    ).count()
)

print("Total Columns:", len(fact_reviews_final_df.columns))

assert total_records == 99224
assert unique_reviews == 99224
assert duplicate_records == 0

print("==============================================")
print("FACT REVIEWS VALIDATION COMPLETED")

========== FACT REVIEWS VALIDATION ==========
Total Records: 99224
Unique Review-Order Combinations: 99224
Duplicate Records: 0
NULL Review IDs: 0
NULL Order IDs: 0
NULL Review Scores: 0
Total Columns: 21
FACT REVIEWS VALIDATION COMPLETED


In [0]:
(
    fact_reviews_final_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.fact_reviews")
)

# Verify saved table
saved_fact_reviews_df = spark.table("workspace.gold.fact_reviews")

print("FACT REVIEWS SAVED SUCCESSFULLY!")
print("Table: workspace.gold.fact_reviews")
print("Records:", saved_fact_reviews_df.count())
print("Columns:", len(saved_fact_reviews_df.columns))

saved_fact_reviews_df.printSchema()

FACT REVIEWS SAVED SUCCESSFULLY!
Table: workspace.gold.fact_reviews
Records: 99224
Columns: 21
root
 |-- customer_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- review_date: date (nullable = true)
 |-- review_date_key: integer (nullable = true)
 |-- purchase_date: date (nullable = true)
 |-- purchase_date_key: integer (nullable = true)
 |-- customer_geo_key: long (n

In [0]:
silver_mql_df = spark.table(
    "workspace.silver.silver_marketing_qualified_leads"
)

silver_deals_df = spark.table(
    "workspace.silver.silver_closed_deals"
)

print("========== MARKETING QUALIFIED LEADS ==========")
print("Records:", silver_mql_df.count())
silver_mql_df.printSchema()
display(silver_mql_df.limit(5))

print("========== CLOSED DEALS ==========")
print("Records:", silver_deals_df.count())
silver_deals_df.printSchema()
display(silver_deals_df.limit(5))

========== MARKETING QUALIFIED LEADS ==========
Records: 8000
root
 |-- mql_id: string (nullable = true)
 |-- first_contact_date: date (nullable = true)
 |-- landing_page_id: string (nullable = true)
 |-- origin: string (nullable = true)



mql_id,first_contact_date,landing_page_id,origin
dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social
8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search
b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search
6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email
5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search


========== CLOSED DEALS ==========
Records: 842
root
 |-- mql_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- sdr_id: string (nullable = true)
 |-- sr_id: string (nullable = true)
 |-- won_date: timestamp (nullable = true)
 |-- business_segment: string (nullable = true)
 |-- lead_type: string (nullable = true)
 |-- lead_behaviour_profile: string (nullable = true)
 |-- has_company: boolean (nullable = true)
 |-- has_gtin: boolean (nullable = true)
 |-- average_stock: string (nullable = true)
 |-- business_type: string (nullable = true)
 |-- declared_product_catalog_size: double (nullable = true)
 |-- declared_monthly_revenue: double (nullable = true)



mql_id,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue
5420aad7fec3549a85876ba1c529bd84,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26T19:58:54.000Z,pet,online_medium,cat,null,null,null,reseller,null,0.0
a555fb36b9368110ede0f043dfc3b9a0,bbb7d7893a450660432ea6652310ebb7,09285259593c61296eef10c734121d5b,d3d1e91a157ea7f90548eef82f1955e3,2018-05-08T20:17:59.000Z,car_accessories,industry,eagle,null,null,null,reseller,null,0.0
327174d3648a2d047e8940d7d15204ca,612170e34b97004b3ba37eae81836b4c,b90f87164b5f8c2cfa5c8572834dbe3f,6565aa9ce3178a5caf6171827af3a9ba,2018-06-05T17:27:23.000Z,home_appliances,online_big,cat,null,null,null,reseller,null,0.0
f5fee8f7da74f4887f5bcae2bafb6dd6,21e1781e36faf92725dde4730a88ca0f,56bf83c4bb35763a51c2baab501b4c67,d3d1e91a157ea7f90548eef82f1955e3,2018-01-17T13:51:03.000Z,food_drink,online_small,null,null,null,null,reseller,null,0.0
ffe640179b554e295c167a2f6be528e0,ed8cb7b190ceb6067227478e48cf8dde,4b339f9567d060bcea4f5136b9f5949e,d3d1e91a157ea7f90548eef82f1955e3,2018-07-03T20:17:45.000Z,home_appliances,industry,wolf,null,null,null,manufacturer,null,0.0


In [0]:
print("========== MQL UNIQUENESS ==========")

print("Total MQL Records:", silver_mql_df.count())

print(
    "Unique MQL IDs:",
    silver_mql_df.select("mql_id").distinct().count()
)

print("========== CLOSED DEAL UNIQUENESS ==========")

print("Total Deal Records:", silver_deals_df.count())

print(
    "Unique Deal MQL IDs:",
    silver_deals_df.select("mql_id").distinct().count()
)

print("========== DEAL MATCHING ==========")

matched_mqls = (
    silver_mql_df
    .join(
        silver_deals_df.select("mql_id"),
        on="mql_id",
        how="inner"
    )
    .select("mql_id")
    .distinct()
    .count()
)

print("MQLs With Closed Deals:", matched_mqls)

print(
    "MQLs Without Closed Deals:",
    silver_mql_df.count() - matched_mqls
)

========== MQL UNIQUENESS ==========
Total MQL Records: 8000
Unique MQL IDs: 8000
========== CLOSED DEAL UNIQUENESS ==========
Total Deal Records: 842
Unique Deal MQL IDs: 842
========== DEAL MATCHING ==========
MQLs With Closed Deals: 842
MQLs Without Closed Deals: 7158


In [0]:
fact_marketing_base_df = (
    silver_mql_df.alias("m")
    .join(
        silver_deals_df.alias("d"),
        on="mql_id",
        how="left"
    )
    .select(
        F.col("m.mql_id"),
        F.col("m.first_contact_date"),
        F.col("m.landing_page_id"),
        F.col("m.origin"),

        F.col("d.seller_id"),
        F.col("d.sdr_id"),
        F.col("d.sr_id"),
        F.col("d.won_date"),
        F.col("d.business_segment"),
        F.col("d.lead_type"),
        F.col("d.lead_behaviour_profile"),
        F.col("d.has_company"),
        F.col("d.has_gtin"),
        F.col("d.average_stock"),
        F.col("d.business_type"),
        F.col("d.declared_product_catalog_size"),
        F.col("d.declared_monthly_revenue")
    )
    .withColumn(
        "is_converted",
        F.col("won_date").isNotNull()
    )
)

print("Marketing Base Records:", fact_marketing_base_df.count())

print(
    "Unique MQL IDs:",
    fact_marketing_base_df.select("mql_id").distinct().count()
)

print("Total Columns:", len(fact_marketing_base_df.columns))

display(fact_marketing_base_df.limit(5))

Marketing Base Records: 8000
Unique MQL IDs: 8000
Total Columns: 18


mql_id,first_contact_date,landing_page_id,origin,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue,is_converted
dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social,null,null,null,null,null,null,null,null,null,null,null,null,null,false
8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search,null,null,null,null,null,null,null,null,null,null,null,null,null,false
b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search,null,null,null,null,null,null,null,null,null,null,null,null,null,false
6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email,null,null,null,null,null,null,null,null,null,null,null,null,null,false
5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26T19:58:54.000Z,pet,online_medium,cat,null,null,null,reseller,null,0.0,true


In [0]:
dim_sellers_df = spark.table("workspace.gold.dim_sellers")
dim_geo_df = spark.table("workspace.gold.dim_geolocation")
dim_date_df = spark.table("workspace.gold.dim_date")

fact_marketing_enriched_df = (
    fact_marketing_base_df.alias("m")

    # Seller dimension
    .join(
        dim_sellers_df.alias("s"),
        F.col("m.seller_id") == F.col("s.seller_id"),
        how="left"
    )

    # First contact date dimension
    .join(
        dim_date_df.alias("fd"),
        F.col("m.first_contact_date") == F.col("fd.full_date"),
        how="left"
    )

    # Deal won date dimension
    .join(
        dim_date_df.alias("wd"),
        F.to_date(F.col("m.won_date")) == F.col("wd.full_date"),
        how="left"
    )

    # Seller geolocation dimension
    .join(
        dim_geo_df.alias("g"),
        F.col("s.seller_zip_code_prefix") ==
        F.col("g.geolocation_zip_code_prefix"),
        how="left"
    )

    .select(
        F.col("m.*"),
        F.col("s.seller_zip_code_prefix"),
        F.col("s.seller_city"),
        F.col("s.seller_state"),
        F.col("fd.date_key").alias("first_contact_date_key"),
        F.col("wd.date_key").alias("won_date_key"),
        F.col("g.geo_key").alias("seller_geo_key")
    )
)

print(
    "Marketing Enriched Records:",
    fact_marketing_enriched_df.count()
)

print(
    "Unique MQL IDs:",
    fact_marketing_enriched_df
    .select("mql_id")
    .distinct()
    .count()
)

print("Total Columns:", len(fact_marketing_enriched_df.columns))

display(fact_marketing_enriched_df.limit(5))

Marketing Enriched Records: 8000
Unique MQL IDs: 8000
Total Columns: 24


mql_id,first_contact_date,landing_page_id,origin,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue,is_converted,seller_zip_code_prefix,seller_city,seller_state,first_contact_date_key,won_date_key,seller_geo_key
dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social,null,null,null,null,null,null,null,null,null,null,null,null,null,false,null,null,null,20180201,null,null
8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search,null,null,null,null,null,null,null,null,null,null,null,null,null,false,null,null,null,20171020,null,null
b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search,null,null,null,null,null,null,null,null,null,null,null,null,null,false,null,null,null,20180322,null,null
6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email,null,null,null,null,null,null,null,null,null,null,null,null,null,false,null,null,null,20180122,null,null
5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26T19:58:54.000Z,pet,online_medium,cat,null,null,null,reseller,null,0.0,true,30421,belo horizonte,MG,20180221,20180226,30421


In [0]:
fact_marketing_enriched_df.agg(
    F.count("*").alias("total_records"),

    F.countDistinct("mql_id").alias("unique_mql_ids"),

    F.sum(
        F.col("first_contact_date_key").isNull().cast("int")
    ).alias("missing_first_contact_date_key"),

    F.sum(
        (F.col("is_converted") == True).cast("int")
    ).alias("converted_leads"),

    F.sum(
        (F.col("is_converted") == False).cast("int")
    ).alias("non_converted_leads"),

    F.sum(
        (
            (F.col("is_converted") == True) &
            F.col("won_date_key").isNull()
        ).cast("int")
    ).alias("converted_without_won_date_key"),

    F.sum(
        (
            (F.col("is_converted") == True) &
            F.col("seller_id").isNull()
        ).cast("int")
    ).alias("converted_without_seller_id"),

    F.sum(
        (
            F.col("seller_id").isNotNull() &
            F.col("seller_zip_code_prefix").isNull()
        ).cast("int")
    ).alias("missing_seller_dimension"),

    F.sum(
        (
            F.col("seller_zip_code_prefix").isNotNull() &
            F.col("seller_geo_key").isNull()
        ).cast("int")
    ).alias("missing_seller_geo_key")
).show()

+-------------+--------------+------------------------------+---------------+-------------------+------------------------------+---------------------------+------------------------+----------------------+
|total_records|unique_mql_ids|missing_first_contact_date_key|converted_leads|non_converted_leads|converted_without_won_date_key|converted_without_seller_id|missing_seller_dimension|missing_seller_geo_key|
+-------------+--------------+------------------------------+---------------+-------------------+------------------------------+---------------------------+------------------------+----------------------+
|         8000|          8000|                             0|            842|               7158|                             0|                          0|                     462|                     1|
+-------------+--------------+------------------------------+---------------+-------------------+------------------------------+---------------------------+------------------------

In [0]:
conversion_flag_mismatches = (
    fact_marketing_enriched_df
    .filter(
        F.col("is_converted") != F.col("won_date").isNotNull()
    )
    .count()
)

print("Conversion Flag Mismatches:", conversion_flag_mismatches)

assert conversion_flag_mismatches == 0

print("CONVERSION FLAG VALIDATION PASSED")

Conversion Flag Mismatches: 0
CONVERSION FLAG VALIDATION PASSED


In [0]:
fact_marketing_enriched_df.agg(
    F.count("*").alias("total_records"),
    F.countDistinct("mql_id").alias("unique_mql_ids"),
    F.sum(F.col("first_contact_date_key").isNull().cast("int")).alias("missing_first_contact_date_key"),
    F.sum((F.col("is_converted") == True).cast("int")).alias("converted_leads"),
    F.sum((F.col("is_converted") == False).cast("int")).alias("non_converted_leads"),
    F.sum(((F.col("is_converted") == True) & F.col("won_date_key").isNull()).cast("int")).alias("converted_without_won_date_key"),
    F.sum(((F.col("is_converted") == True) & F.col("seller_id").isNull()).cast("int")).alias("converted_without_seller_id"),
    F.sum((F.col("seller_id").isNotNull() & F.col("seller_zip_code_prefix").isNull()).cast("int")).alias("missing_seller_dimension"),
    F.sum((F.col("seller_zip_code_prefix").isNotNull() & F.col("seller_geo_key").isNull()).cast("int")).alias("missing_seller_geo_key")
).show()

+-------------+--------------+------------------------------+---------------+-------------------+------------------------------+---------------------------+------------------------+----------------------+
|total_records|unique_mql_ids|missing_first_contact_date_key|converted_leads|non_converted_leads|converted_without_won_date_key|converted_without_seller_id|missing_seller_dimension|missing_seller_geo_key|
+-------------+--------------+------------------------------+---------------+-------------------+------------------------------+---------------------------+------------------------+----------------------+
|         8000|          8000|                             0|            842|               7158|                             0|                          0|                     462|                     1|
+-------------+--------------+------------------------------+---------------+-------------------+------------------------------+---------------------------+------------------------

In [0]:
fact_marketing_final_df = (
    fact_marketing_enriched_df
    .withColumn("lead_count", F.lit(1))
    .withColumn(
        "conversion_count",
        F.when(F.col("is_converted"), 1).otherwise(0)
    )
    .withColumn(
        "days_to_conversion",
        F.when(
            F.col("is_converted"),
            F.datediff(
                F.to_date(F.col("won_date")),
                F.col("first_contact_date")
            )
        )
    )
)

display(fact_marketing_final_df)

mql_id,first_contact_date,landing_page_id,origin,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue,is_converted,seller_zip_code_prefix,seller_city,seller_state,first_contact_date_key,won_date_key,seller_geo_key,lead_count,conversion_count,days_to_conversion
dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social,null,null,null,null,null,null,null,null,null,null,null,null,null,false,null,null,null,20180201,null,null,1,0,null
8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search,null,null,null,null,null,null,null,null,null,null,null,null,null,false,null,null,null,20171020,null,null,1,0,null
b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search,null,null,null,null,null,null,null,null,null,null,null,null,null,false,null,null,null,20180322,null,null,1,0,null
6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email,null,null,null,null,null,null,null,null,null,null,null,null,null,false,null,null,null,20180122,null,null,1,0,null
5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26T19:58:54.000Z,pet,online_medium,cat,null,null,null,reseller,null,0.0,true,30421,belo horizonte,MG,20180221,20180226,30421,1,1,5
28bdfd5f057764b54c38770f95c69f2f,2018-01-14,22c29808c4f815213303f8933030604c,organic_search,null,null,null,null,null,null,null,null,null,null,null,null,null,false,null,null,null,20180114,null,null,1,0,null
126a0d10becbaafcb2e72ce6848cf32c,2018-05-15,6a110e795dd487f1cf8d7583671987af,email,null,null,null,null,null,null,null,null,null,null,null,null,null,false,null,null,null,20180515,null,null,1,0,null
f76136f54d14a3345951f25b7932366b,2018-05-24,d51b0d02f063ba1d053db6d97226eec3,email,null,null,null,null,null,null,null,null,null,null,null,null,null,false,null,null,null,20180524,null,null,1,0,null
2f838cade4a6012a6cb1016d1d8d95ed,2017-11-10,aeac92c0f5ae22a04ed3b746cce3a1b6,organic_search,null,null,null,null,null,null,null,null,null,null,null,null,null,false,null,null,null,20171110,null,null,1,0,null
7281942387a1a0c3f72a50a8b0bb0920,2017-12-25,88740e65d5d6b056e0cda098e1ea6313,social,null,null,null,null,null,null,null,null,null,null,null,null,null,false,null,null,null,20171225,null,null,1,0,null


In [0]:
fact_marketing_final_df.agg(
    F.count("*").alias("total_leads"),
    F.sum("lead_count").alias("lead_count"),
    F.sum("conversion_count").alias("total_conversions"),
    F.round(
        F.sum("conversion_count") / F.sum("lead_count") * 100, 2
    ).alias("conversion_rate_pct"),
    F.round(F.avg("days_to_conversion"), 2).alias("avg_days_to_conversion"),
    F.min("days_to_conversion").alias("min_days_to_conversion"),
    F.max("days_to_conversion").alias("max_days_to_conversion"),
    F.sum((F.col("days_to_conversion") < 0).cast("int")).alias("negative_conversion_days")
).show()

+-----------+----------+-----------------+-------------------+----------------------+----------------------+----------------------+------------------------+
|total_leads|lead_count|total_conversions|conversion_rate_pct|avg_days_to_conversion|min_days_to_conversion|max_days_to_conversion|negative_conversion_days|
+-----------+----------+-----------------+-------------------+----------------------+----------------------+----------------------+------------------------+
|       8000|      8000|              842|              10.53|                 48.44|                    -2|                   427|                       1|
+-----------+----------+-----------------+-------------------+----------------------+----------------------+----------------------+------------------------+



In [0]:
display(
    fact_marketing_final_df
    .filter(F.col("days_to_conversion") < 0)
    .select(
        "mql_id",
        "first_contact_date",
        "won_date",
        "days_to_conversion",
        "seller_id",
        "origin",
        "landing_page_id"
    )
)

mql_id,first_contact_date,won_date,days_to_conversion,seller_id,origin,landing_page_id
b91cf8812365f50ff4bda4bcd6206b05,2018-03-08,2018-03-06T19:38:55.000Z,-2,5e063e85d44b0f5c3e6ec3131103a57e,organic_search,4aab772aeb9833c170d8ea5413e02eac


In [0]:
fact_marketing_final_df = (
    fact_marketing_final_df
    .withColumn(
        "days_to_conversion",
        F.when(
            F.col("is_converted") &
            (F.datediff(
                F.to_date("won_date"),
                F.col("first_contact_date")
            ) >= 0),
            F.datediff(
                F.to_date("won_date"),
                F.col("first_contact_date")
            )
        )
    )
)

In [0]:
fact_marketing_final_df.agg(
    F.count("*").alias("total_leads"),
    F.sum("lead_count").alias("lead_count"),
    F.sum("conversion_count").alias("total_conversions"),
    F.round(
        F.sum("conversion_count") / F.sum("lead_count") * 100, 2
    ).alias("conversion_rate_pct"),
    F.round(F.avg("days_to_conversion"), 2).alias("avg_days_to_conversion"),
    F.min("days_to_conversion").alias("min_days_to_conversion"),
    F.max("days_to_conversion").alias("max_days_to_conversion"),
    F.sum(
        (F.col("days_to_conversion") < 0).cast("int")
    ).alias("negative_conversion_days")
).show()

+-----------+----------+-----------------+-------------------+----------------------+----------------------+----------------------+------------------------+
|total_leads|lead_count|total_conversions|conversion_rate_pct|avg_days_to_conversion|min_days_to_conversion|max_days_to_conversion|negative_conversion_days|
+-----------+----------+-----------------+-------------------+----------------------+----------------------+----------------------+------------------------+
|       8000|      8000|              842|              10.53|                  48.5|                     0|                   427|                       0|
+-----------+----------+-----------------+-------------------+----------------------+----------------------+----------------------+------------------------+



In [0]:
fact_marketing_final_df.agg(
    F.count("*").alias("total_records"),
    F.countDistinct("mql_id").alias("unique_mql_ids"),
    F.sum(F.col("mql_id").isNull().cast("int")).alias("null_mql_ids"),
    F.sum(F.col("first_contact_date").isNull().cast("int")).alias("null_first_contact_dates"),
    F.sum("lead_count").alias("total_lead_count"),
    F.sum("conversion_count").alias("total_conversion_count"),
    F.sum(
        (
            F.col("is_converted") !=
            (F.col("conversion_count") == 1)
        ).cast("int")
    ).alias("conversion_flag_mismatches"),
    F.sum(
        (
            F.col("is_converted") &
            F.col("won_date").isNull()
        ).cast("int")
    ).alias("converted_without_won_date"),
    F.sum(
        (F.col("days_to_conversion") < 0).cast("int")
    ).alias("negative_conversion_days")
).show()

+-------------+--------------+------------+------------------------+----------------+----------------------+--------------------------+--------------------------+------------------------+
|total_records|unique_mql_ids|null_mql_ids|null_first_contact_dates|total_lead_count|total_conversion_count|conversion_flag_mismatches|converted_without_won_date|negative_conversion_days|
+-------------+--------------+------------+------------------------+----------------+----------------------+--------------------------+--------------------------+------------------------+
|         8000|          8000|           0|                       0|            8000|                   842|                         0|                         0|                       0|
+-------------+--------------+------------+------------------------+----------------+----------------------+--------------------------+--------------------------+------------------------+



In [0]:
(
    fact_marketing_final_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.fact_marketing_conversion")
)

In [0]:
saved_marketing_df = spark.table(
    "workspace.gold.fact_marketing_conversion"
)

print("Total Records:", saved_marketing_df.count())
print("Total Columns:", len(saved_marketing_df.columns))

saved_marketing_df.printSchema()

Total Records: 8000
Total Columns: 27
root
 |-- mql_id: string (nullable = true)
 |-- first_contact_date: date (nullable = true)
 |-- landing_page_id: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- sdr_id: string (nullable = true)
 |-- sr_id: string (nullable = true)
 |-- won_date: timestamp (nullable = true)
 |-- business_segment: string (nullable = true)
 |-- lead_type: string (nullable = true)
 |-- lead_behaviour_profile: string (nullable = true)
 |-- has_company: boolean (nullable = true)
 |-- has_gtin: boolean (nullable = true)
 |-- average_stock: string (nullable = true)
 |-- business_type: string (nullable = true)
 |-- declared_product_catalog_size: double (nullable = true)
 |-- declared_monthly_revenue: double (nullable = true)
 |-- is_converted: boolean (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- f

In [0]:
from pyspark.sql import functions as F

# Load the four Gold fact tables
sales_df = spark.table("workspace.gold.fact_sales")
payments_df = spark.table("workspace.gold.fact_payments")
reviews_df = spark.table("workspace.gold.fact_reviews")
marketing_df = spark.table("workspace.gold.fact_marketing_conversion")

# 1. Sales KPIs
sales_kpis = sales_df.agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.sum("order_item_count").alias("total_order_items"),
    F.round(F.sum("price"), 2).alias("product_sales_brl"),
    F.round(F.sum("freight_value"), 2).alias("freight_value_brl"),
    F.round(F.sum("total_item_value"), 2).alias("sales_including_freight_brl"),
    F.round(
        F.sum("total_item_value") / F.countDistinct("order_id"), 2
    ).alias("average_order_value_brl"),
    F.countDistinct("customer_unique_id").alias("unique_customers")
)

# 2. Payment KPIs
payment_kpis = payments_df.agg(
    F.sum("payment_count").alias("total_payment_records"),
    F.round(F.sum("payment_value"), 2).alias("total_payment_value_brl"),
    F.sum(F.col("is_zero_payment").cast("int")).alias("zero_payment_records")
)

# 3. Review KPIs
review_kpis = reviews_df.agg(
    F.sum("review_count").alias("total_reviews"),
    F.round(F.avg("review_score"), 2).alias("average_review_score"),
    F.sum(F.col("is_positive_review").cast("int")).alias("positive_reviews"),
    F.round(
        F.sum(F.col("is_positive_review").cast("int"))
        / F.sum("review_count") * 100,
        2
    ).alias("positive_review_rate_pct")
)

# 4. Marketing KPIs
marketing_kpis = marketing_df.agg(
    F.sum("lead_count").alias("total_leads"),
    F.sum("conversion_count").alias("total_conversions"),
    F.round(
        F.sum("conversion_count") / F.sum("lead_count") * 100, 2
    ).alias("conversion_rate_pct"),
    F.round(F.avg("days_to_conversion"), 2).alias("average_days_to_conversion")
)

# Combine the four one-row KPI summaries
executive_kpi_df = (
    sales_kpis
    .crossJoin(payment_kpis)
    .crossJoin(review_kpis)
    .crossJoin(marketing_kpis)
)

display(executive_kpi_df)

total_orders,total_order_items,product_sales_brl,freight_value_brl,sales_including_freight_brl,average_order_value_brl,unique_customers,total_payment_records,total_payment_value_brl,zero_payment_records,total_reviews,average_review_score,positive_reviews,positive_review_rate_pct,total_leads,total_conversions,conversion_rate_pct,average_days_to_conversion
98666,112650,1.35916437E7,2251909.54,1.584355324E7,160.58,95420,103886,1.600887212E7,9,99224,4.09,76470,77.07,8000,842,10.53,48.5


In [0]:
executive_kpi_final_df = (
    executive_kpi_df
    .withColumn("snapshot_created_at", F.current_timestamp())
)

(
    executive_kpi_final_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.kpi_executive_summary")
)

print("Executive KPI summary saved successfully!")

Executive KPI summary saved successfully!


In [0]:
monthly_sales_trend_df = (
    sales_df
    .withColumn(
        "year_month",
        F.date_format("purchase_date", "yyyy-MM")
    )
    .groupBy("year_month")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("order_item_count").alias("total_order_items"),
        F.round(F.sum("price"), 2).alias("product_sales_brl"),
        F.round(F.sum("freight_value"), 2).alias("freight_value_brl"),
        F.round(F.sum("total_item_value"), 2).alias("sales_including_freight_brl")
    )
    .orderBy("year_month")
)

display(monthly_sales_trend_df)

year_month,total_orders,total_order_items,product_sales_brl,freight_value_brl,sales_including_freight_brl
2016-09,3,6,267.36,87.39,354.75
2016-10,308,363,49507.66,7301.18,56808.84
2016-12,1,1,10.9,8.72,19.62
2017-01,789,955,120312.87,16875.62,137188.49
2017-02,1733,1951,247303.02,38977.6,286280.62
2017-03,2641,3000,374344.3,57704.29,432048.59
2017-04,2391,2684,359927.23,52495.01,412422.24
2017-05,3660,4136,506071.14,80119.81,586190.95
2017-06,3217,3583,433038.6,69924.44,502963.04
2017-07,3969,4519,498031.48,86940.14,584971.62


In [0]:
monthly_sales_trend_df.agg(
    F.count("*").alias("months_with_sales"),
    F.sum("total_orders").alias("monthly_total_orders"),
    F.sum("total_order_items").alias("monthly_total_items"),
    F.round(F.sum("product_sales_brl"), 2).alias("monthly_product_sales"),
    F.round(F.sum("freight_value_brl"), 2).alias("monthly_freight"),
    F.round(F.sum("sales_including_freight_brl"), 2).alias("monthly_total_sales")
).show()

+-----------------+--------------------+-------------------+---------------------+---------------+-------------------+
|months_with_sales|monthly_total_orders|monthly_total_items|monthly_product_sales|monthly_freight|monthly_total_sales|
+-----------------+--------------------+-------------------+---------------------+---------------+-------------------+
|               24|               98666|             112650|         1.35916437E7|     2251909.54|      1.584355324E7|
+-----------------+--------------------+-------------------+---------------------+---------------+-------------------+



In [0]:
(
    monthly_sales_trend_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.kpi_monthly_sales")
)

print("Monthly sales KPI table saved successfully!")

Monthly sales KPI table saved successfully!


In [0]:
monthly_payment_trend_df = (
    payments_df
    .withColumn(
        "year_month",
        F.date_format("purchase_date", "yyyy-MM")
    )
    .groupBy("year_month")
    .agg(
        F.sum("payment_count").alias("total_payment_records"),
        F.countDistinct("order_id").alias("orders_with_payments"),
        F.round(F.sum("payment_value"), 2).alias("total_payment_value_brl"),
        F.round(F.avg("payment_value"), 2).alias("average_payment_brl"),
        F.sum(
            F.col("is_zero_payment").cast("int")
        ).alias("zero_payment_records")
    )
    .orderBy("year_month")
)

display(monthly_payment_trend_df)

year_month,total_payment_records,orders_with_payments,total_payment_value_brl,average_payment_brl,zero_payment_records
2016-09,3,3,252.24,84.08,0
2016-10,342,324,59090.48,172.78,0
2016-12,1,1,19.62,19.62,0
2017-01,850,800,138488.04,162.93,0
2017-02,1886,1780,291908.01,154.78,0
2017-03,2837,2682,449863.6,158.57,0
2017-04,2571,2404,417788.03,162.5,2
2017-05,3944,3700,592918.82,150.33,1
2017-06,3436,3245,511276.38,148.8,1
2017-07,4317,4026,592382.92,137.22,0


In [0]:
monthly_payment_trend_df.agg(
    F.count("*").alias("months_with_payments"),
    F.sum("total_payment_records").alias("monthly_payment_records"),
    F.sum("orders_with_payments").alias("monthly_orders_with_payments"),
    F.round(
        F.sum("total_payment_value_brl"), 2
    ).alias("monthly_total_payment_value"),
    F.sum("zero_payment_records").alias("monthly_zero_payment_records")
).show()

+--------------------+-----------------------+----------------------------+---------------------------+----------------------------+
|months_with_payments|monthly_payment_records|monthly_orders_with_payments|monthly_total_payment_value|monthly_zero_payment_records|
+--------------------+-----------------------+----------------------------+---------------------------+----------------------------+
|                  25|                 103886|                       99440|              1.600887212E7|                           9|
+--------------------+-----------------------+----------------------------+---------------------------+----------------------------+



In [0]:
(
    monthly_payment_trend_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.kpi_monthly_payments")
)

print("Monthly payment KPI table saved successfully!")

Monthly payment KPI table saved successfully!


In [0]:
monthly_review_trend_df = (
    reviews_df
    .withColumn(
        "year_month",
        F.date_format("review_date", "yyyy-MM")
    )
    .groupBy("year_month")
    .agg(
        F.sum("review_count").alias("total_reviews"),
        F.round(F.avg("review_score"), 2).alias("average_review_score"),
        F.sum(
            F.when(F.col("review_score") >= 4, 1).otherwise(0)
        ).alias("positive_reviews"),
        F.sum(
            F.when(F.col("review_score") <= 2, 1).otherwise(0)
        ).alias("negative_reviews"),
        F.sum(
            F.when(F.col("review_score") == 3, 1).otherwise(0)
        ).alias("neutral_reviews"),
        F.round(
            F.sum(
                F.when(F.col("review_score") >= 4, 1).otherwise(0)
            ) / F.sum("review_count") * 100,
            2
        ).alias("positive_review_rate_pct")
    )
    .orderBy("year_month")
)

display(monthly_review_trend_df)

year_month,total_reviews,average_review_score,positive_reviews,negative_reviews,neutral_reviews,positive_review_rate_pct
2016-10,179,4.06,137,29,13,76.54
2016-11,101,3.19,55,38,8,54.46
2016-12,45,2.36,15,29,1,33.33
2017-01,239,4.34,196,20,23,82.01
2017-02,1413,4.28,1158,135,120,81.95
2017-03,2481,4.03,1874,390,217,75.53
2017-04,2055,4.04,1546,315,194,75.23
2017-05,3710,4.1,2870,511,329,77.36
2017-06,3440,4.13,2679,452,309,77.88
2017-07,3503,4.18,2780,422,301,79.36


In [0]:
monthly_review_trend_df.agg(
    F.count("*").alias("months_with_reviews"),
    F.sum("total_reviews").alias("monthly_total_reviews"),
    F.sum("positive_reviews").alias("monthly_positive_reviews"),
    F.sum("negative_reviews").alias("monthly_negative_reviews"),
    F.sum("neutral_reviews").alias("monthly_neutral_reviews"),
    F.round(
        F.sum("positive_reviews") / F.sum("total_reviews") * 100, 2
    ).alias("overall_positive_rate_pct")
).show()

+-------------------+---------------------+------------------------+------------------------+-----------------------+-------------------------+
|months_with_reviews|monthly_total_reviews|monthly_positive_reviews|monthly_negative_reviews|monthly_neutral_reviews|overall_positive_rate_pct|
+-------------------+---------------------+------------------------+------------------------+-----------------------+-------------------------+
|                 23|                99224|                   76470|                   14575|                   8179|                    77.07|
+-------------------+---------------------+------------------------+------------------------+-----------------------+-------------------------+



In [0]:
(
    monthly_review_trend_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.kpi_monthly_reviews")
)

print("Monthly review KPI table saved successfully!")

Monthly review KPI table saved successfully!


In [0]:
monthly_marketing_trend_df = (
    marketing_df
    .withColumn(
        "year_month",
        F.date_format("first_contact_date", "yyyy-MM")
    )
    .groupBy("year_month")
    .agg(
        F.sum("lead_count").alias("total_leads"),
        F.sum("conversion_count").alias("total_conversions"),
        F.sum(
            F.when(F.col("is_converted") == False, 1).otherwise(0)
        ).alias("non_converted_leads"),
        F.round(
            F.sum("conversion_count") / F.sum("lead_count") * 100,
            2
        ).alias("conversion_rate_pct"),
        F.round(
            F.avg("days_to_conversion"), 2
        ).alias("average_days_to_conversion")
    )
    .orderBy("year_month")
)

display(monthly_marketing_trend_df)

year_month,total_leads,total_conversions,non_converted_leads,conversion_rate_pct,average_days_to_conversion
2017-06,4,0,4,0.0,null
2017-07,239,2,237,0.84,398.0
2017-08,386,9,377,2.33,296.0
2017-09,312,7,305,2.24,283.71
2017-10,416,14,402,3.37,240.21
2017-11,445,18,427,4.04,155.11
2017-12,200,11,189,5.5,122.36
2018-01,1141,152,989,13.32,43.65
2018-02,1028,149,879,14.49,42.28
2018-03,1174,167,1007,14.22,37.85


In [0]:
monthly_marketing_trend_df.agg(
    F.count("*").alias("months_with_leads"),
    F.sum("total_leads").alias("monthly_total_leads"),
    F.sum("total_conversions").alias("monthly_total_conversions"),
    F.sum("non_converted_leads").alias("monthly_non_converted_leads"),
    F.round(
        F.sum("total_conversions") /
        F.sum("total_leads") * 100, 2
    ).alias("overall_conversion_rate_pct")
).show()

+-----------------+-------------------+-------------------------+---------------------------+---------------------------+
|months_with_leads|monthly_total_leads|monthly_total_conversions|monthly_non_converted_leads|overall_conversion_rate_pct|
+-----------------+-------------------+-------------------------+---------------------------+---------------------------+
|               12|               8000|                      842|                       7158|                      10.53|
+-----------------+-------------------+-------------------------+---------------------------+---------------------------+



In [0]:
(
    monthly_marketing_trend_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.kpi_monthly_marketing")
)

print("Monthly marketing KPI table saved successfully!")

Monthly marketing KPI table saved successfully!


In [0]:
sales_by_category_df = (
    sales_df
    .withColumn(
        "product_category",
        F.coalesce(
            F.col("product_category_name_english"),
            F.lit("Unknown / untranslated")
        )
    )
    .groupBy("product_category")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("order_item_count").alias("total_items_sold"),
        F.round(F.sum("price"), 2).alias("product_sales_brl"),
        F.round(F.sum("freight_value"), 2).alias("freight_brl"),
        F.round(F.sum("total_item_value"), 2).alias("total_sales_brl")
    )
    .orderBy(F.desc("product_sales_brl"))
)

display(sales_by_category_df)

product_category,total_orders,total_items_sold,product_sales_brl,freight_brl,total_sales_brl
health_beauty,8836,9670,1258681.34,182566.73,1441248.07
watches_gifts,5624,5991,1205005.68,100535.93,1305541.61
bed_bath_table,9417,11115,1036988.68,204693.04,1241681.72
sports_leisure,7720,8641,988048.97,168607.51,1156656.48
computers_accessories,6689,7827,911954.32,147318.08,1059272.4
furniture_decor,6449,8334,729762.49,172749.3,902511.79
cool_stuff,3632,3796,635290.85,84039.1,719329.95
housewares,5884,6964,632248.66,146149.11,778397.77
auto,3897,4235,592720.11,92664.21,685384.32
garden_tools,3518,4347,485256.46,98962.75,584219.21


In [0]:
sales_by_category_df.agg(
    F.count("*").alias("total_categories"),
    F.sum("total_items_sold").alias("total_items"),
    F.round(F.sum("product_sales_brl"), 2).alias("total_product_sales"),
    F.round(F.sum("freight_brl"), 2).alias("total_freight"),
    F.round(F.sum("total_sales_brl"), 2).alias("total_sales")
).show()

+----------------+-----------+-------------------+-------------+-------------+
|total_categories|total_items|total_product_sales|total_freight|  total_sales|
+----------------+-----------+-------------------+-------------+-------------+
|              72|     112650|       1.35916437E7|   2251909.54|1.584355324E7|
+----------------+-----------+-------------------+-------------+-------------+



In [0]:
(
    sales_by_category_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.kpi_sales_by_category")
)

print("Sales by category KPI table saved successfully!")

Sales by category KPI table saved successfully!


In [0]:
top_10_categories_df = (
    sales_by_category_df
    .filter(F.col("product_category") != "Unknown / untranslated")
    .orderBy(F.desc("product_sales_brl"))
    .limit(10)
)

display(top_10_categories_df)

product_category,total_orders,total_items_sold,product_sales_brl,freight_brl,total_sales_brl
health_beauty,8836,9670,1258681.34,182566.73,1441248.07
watches_gifts,5624,5991,1205005.68,100535.93,1305541.61
bed_bath_table,9417,11115,1036988.68,204693.04,1241681.72
sports_leisure,7720,8641,988048.97,168607.51,1156656.48
computers_accessories,6689,7827,911954.32,147318.08,1059272.4
furniture_decor,6449,8334,729762.49,172749.3,902511.79
cool_stuff,3632,3796,635290.85,84039.1,719329.95
housewares,5884,6964,632248.66,146149.11,778397.77
auto,3897,4235,592720.11,92664.21,685384.32
garden_tools,3518,4347,485256.46,98962.75,584219.21


In [0]:
(
    top_10_categories_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.kpi_top_10_categories")
)

print("Top 10 categories KPI table saved successfully!")

Top 10 categories KPI table saved successfully!


In [0]:
sales_by_state_df = (
    sales_df
    .withColumn(
        "customer_state",
        F.coalesce(F.col("customer_state"), F.lit("Unknown"))
    )
    .groupBy("customer_state")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.countDistinct("customer_unique_id").alias("unique_customers"),
        F.sum("order_item_count").alias("total_items_sold"),
        F.round(F.sum("price"), 2).alias("product_sales_brl"),
        F.round(F.sum("freight_value"), 2).alias("freight_brl"),
        F.round(F.sum("total_item_value"), 2).alias("total_sales_brl")
    )
    .orderBy(F.desc("product_sales_brl"))
)

display(sales_by_state_df)

customer_state,total_orders,unique_customers,total_items_sold,product_sales_brl,freight_brl,total_sales_brl
SP,41375,39981,47449,5202955.05,718723.07,5921678.12
RJ,12762,12303,14579,1824092.67,305589.31,2129681.98
MG,11544,11178,13129,1585308.03,270853.46,1856161.49
RS,5432,5249,6235,750304.02,135522.74,885826.76
PR,4998,4840,5740,683083.76,117851.68,800935.44
SC,3612,3513,4176,520553.34,89660.26,610213.6
BA,3358,3257,3799,511349.99,100156.68,611506.67
DF,2125,2062,2406,302603.94,50625.5,353229.44
GO,2007,1942,2333,294591.95,53114.98,347706.93
ES,2025,1956,2256,275037.31,49764.6,324801.91


In [0]:
sales_by_state_df.agg(
    F.count("*").alias("total_states"),
    F.sum("total_orders").alias("total_orders"),
    F.sum("total_items_sold").alias("total_items"),
    F.round(F.sum("product_sales_brl"), 2).alias("total_product_sales"),
    F.round(F.sum("freight_brl"), 2).alias("total_freight"),
    F.round(F.sum("total_sales_brl"), 2).alias("total_sales")
).show()

+------------+------------+-----------+-------------------+-------------+-------------+
|total_states|total_orders|total_items|total_product_sales|total_freight|  total_sales|
+------------+------------+-----------+-------------------+-------------+-------------+
|          27|       98666|     112650|       1.35916437E7|   2251909.54|1.584355324E7|
+------------+------------+-----------+-------------------+-------------+-------------+



In [0]:
(
    sales_by_state_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.kpi_sales_by_state")
)

print("State-wise sales KPI table saved successfully!")

State-wise sales KPI table saved successfully!


In [0]:
reviews_by_state_df = (
    reviews_df
    .withColumn(
        "customer_state",
        F.coalesce(F.col("customer_state"), F.lit("Unknown"))
    )
    .groupBy("customer_state")
    .agg(
        F.sum("review_count").alias("total_reviews"),
        F.round(F.avg("review_score"), 2).alias("average_rating"),

        F.sum(F.when(F.col("review_score") == 1, 1).otherwise(0))
            .alias("rating_1_count"),
        F.sum(F.when(F.col("review_score") == 2, 1).otherwise(0))
            .alias("rating_2_count"),
        F.sum(F.when(F.col("review_score") == 3, 1).otherwise(0))
            .alias("rating_3_count"),
        F.sum(F.when(F.col("review_score") == 4, 1).otherwise(0))
            .alias("rating_4_count"),
        F.sum(F.when(F.col("review_score") == 5, 1).otherwise(0))
            .alias("rating_5_count"),

        F.sum(F.when(F.col("review_score") >= 4, 1).otherwise(0))
            .alias("positive_reviews"),
        F.sum(F.when(F.col("review_score") <= 2, 1).otherwise(0))
            .alias("negative_reviews"),
        F.sum(F.when(F.col("review_score") == 3, 1).otherwise(0))
            .alias("neutral_reviews"),

        F.round(
            F.sum(F.when(F.col("review_score") >= 4, 1).otherwise(0))
            / F.sum("review_count") * 100,
            2
        ).alias("positive_review_rate_pct")
    )
    .orderBy(F.desc("total_reviews"))
)

display(reviews_by_state_df)

customer_state,total_reviews,average_rating,rating_1_count,rating_2_count,rating_3_count,rating_4_count,rating_5_count,positive_reviews,negative_reviews,neutral_reviews,positive_review_rate_pct
SP,41690,4.17,4054,1211,3299,7991,25135,33126,5265,3299,79.46
RJ,12765,3.87,2183,464,1050,2137,6931,9068,2647,1050,71.04
MG,11625,4.14,1207,339,969,2259,6851,9110,1546,969,78.37
RS,5483,4.13,560,172,449,1098,3204,4302,732,449,78.46
PR,5038,4.18,473,156,381,1009,3019,4028,629,381,79.95
SC,3623,4.07,413,119,321,712,2058,2770,532,321,76.46
BA,3357,3.86,504,130,337,744,1642,2386,634,337,71.08
DF,2148,4.06,255,68,170,445,1210,1655,323,170,77.05
GO,2024,4.04,233,67,191,423,1110,1533,300,191,75.74
ES,2016,4.04,243,57,182,425,1109,1534,300,182,76.09


In [0]:
reviews_by_state_df.agg(
    F.count("*").alias("total_states"),
    F.sum("total_reviews").alias("total_reviews"),
    F.sum("rating_1_count").alias("rating_1"),
    F.sum("rating_2_count").alias("rating_2"),
    F.sum("rating_3_count").alias("rating_3"),
    F.sum("rating_4_count").alias("rating_4"),
    F.sum("rating_5_count").alias("rating_5"),
    F.sum("positive_reviews").alias("positive_reviews"),
    F.sum("negative_reviews").alias("negative_reviews"),
    F.sum("neutral_reviews").alias("neutral_reviews")
).show()

+------------+-------------+--------+--------+--------+--------+--------+----------------+----------------+---------------+
|total_states|total_reviews|rating_1|rating_2|rating_3|rating_4|rating_5|positive_reviews|negative_reviews|neutral_reviews|
+------------+-------------+--------+--------+--------+--------+--------+----------------+----------------+---------------+
|          27|        99224|   11424|    3151|    8179|   19142|   57328|           76470|           14575|           8179|
+------------+-------------+--------+--------+--------+--------+--------+----------------+----------------+---------------+



In [0]:
(
    reviews_by_state_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.kpi_reviews_by_state")
)

print("State-wise review KPI table saved successfully!")

State-wise review KPI table saved successfully!


In [0]:
sales_by_seller_state_df = (
    sales_df
    .withColumn(
        "seller_state",
        F.coalesce(F.col("seller_state"), F.lit("Unknown"))
    )
    .groupBy("seller_state")
    .agg(
        F.countDistinct("seller_id").alias("active_sellers"),
        F.countDistinct("order_id").alias("orders_handled"),
        F.sum("order_item_count").alias("total_items_sold"),
        F.round(F.sum("price"), 2).alias("product_sales_brl"),
        F.round(F.sum("freight_value"), 2).alias("freight_brl"),
        F.round(F.sum("total_item_value"), 2).alias("total_sales_brl")
    )
    .orderBy(F.desc("product_sales_brl"))
)

display(sales_by_seller_state_df)

seller_state,active_sellers,orders_handled,total_items_sold,product_sales_brl,freight_brl,total_sales_brl
SP,1849,70188,80342,8753396.21,1482487.67,1.023588388E7
PR,349,7673,8671,1261887.21,197013.52,1458900.73
MG,244,7930,8827,1011564.74,212595.06,1224159.8
RJ,171,4353,4818,843984.22,93829.9,937814.12
SC,190,3667,4075,632426.07,106547.06,738973.13
RS,129,1989,2199,378559.54,57243.09,435802.63
BA,19,569,643,285561.56,19700.68,305262.24
DF,30,824,899,97749.48,18494.06,116243.54
PE,9,406,448,91493.85,12392.46,103886.31
GO,40,463,520,66399.21,12565.5,78964.71


In [0]:
sales_by_seller_state_df.agg(
    F.count("*").alias("total_seller_states"),
    F.sum("total_items_sold").alias("total_items"),
    F.round(
        F.sum("product_sales_brl"), 2
    ).alias("total_product_sales"),
    F.round(
        F.sum("freight_brl"), 2
    ).alias("total_freight"),
    F.round(
        F.sum("total_sales_brl"), 2
    ).alias("total_sales")
).show()

+-------------------+-----------+-------------------+-------------+-------------+
|total_seller_states|total_items|total_product_sales|total_freight|  total_sales|
+-------------------+-----------+-------------------+-------------+-------------+
|                 23|     112650|       1.35916437E7|   2251909.54|1.584355324E7|
+-------------------+-----------+-------------------+-------------+-------------+



In [0]:
(
    sales_by_seller_state_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.kpi_sales_by_seller_state")
)

print("Seller-state sales KPI table saved successfully!")

Seller-state sales KPI table saved successfully!


In [0]:
kpi_tables = [
    "kpi_executive_summary",
    "kpi_monthly_sales",
    "kpi_monthly_payments",
    "kpi_monthly_reviews",
    "kpi_monthly_marketing",
    "kpi_sales_by_category",
    "kpi_top_10_categories",
    "kpi_sales_by_state",
    "kpi_reviews_by_state",
    "kpi_sales_by_seller_state"
]

for table_name in kpi_tables:
    df = spark.table(f"workspace.gold.{table_name}")
    print(
        f"{table_name}: "
        f"{df.count():,} records | "
        f"{len(df.columns)} columns"
    )

kpi_executive_summary: 1 records | 19 columns
kpi_monthly_sales: 24 records | 6 columns
kpi_monthly_payments: 25 records | 6 columns
kpi_monthly_reviews: 23 records | 7 columns
kpi_monthly_marketing: 12 records | 6 columns
kpi_sales_by_category: 72 records | 6 columns
kpi_top_10_categories: 10 records | 6 columns
kpi_sales_by_state: 27 records | 7 columns
kpi_reviews_by_state: 27 records | 12 columns
kpi_sales_by_seller_state: 23 records | 7 columns


In [0]:
silver_orders_df = spark.table(
    "workspace.silver.silver_orders"
)

print("Total Orders:", silver_orders_df.count())
print("Total Columns:", len(silver_orders_df.columns))

silver_orders_df.printSchema()

display(silver_orders_df.limit(10))

Total Orders: 99441
Total Columns: 8
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z


In [0]:
from pyspark.sql import functions as F

silver_orders_df.groupBy("order_status").agg(
    F.count("*").alias("total_orders"),
    F.sum(
        F.col("order_delivered_customer_date").isNull()
        .cast("int")
    ).alias("missing_delivery_date"),
    F.sum(
        F.col("order_estimated_delivery_date").isNull()
        .cast("int")
    ).alias("missing_estimated_date"),
    F.sum(
        F.col("order_approved_at").isNull()
        .cast("int")
    ).alias("missing_approval_date")
).orderBy(
    F.desc("total_orders")
).show(truncate=False)

+------------+------------+---------------------+----------------------+---------------------+
|order_status|total_orders|missing_delivery_date|missing_estimated_date|missing_approval_date|
+------------+------------+---------------------+----------------------+---------------------+
|delivered   |96478       |92                   |0                     |14                   |
|shipped     |1107        |1107                 |0                     |0                    |
|canceled    |625         |619                  |0                     |141                  |
|unavailable |609         |609                  |0                     |0                    |
|invoiced    |314         |314                  |0                     |0                    |
|processing  |301         |301                  |0                     |0                    |
|created     |5           |5                    |0                     |5                    |
|approved    |2           |2                    |0

In [0]:
from pyspark.sql import functions as F

# Check unusual delivery-date records
silver_orders_df.filter(
    (
        (F.col("order_status") == "delivered") &
        F.col("order_delivered_customer_date").isNull()
    ) |
    (
        (F.col("order_status") == "canceled") &
        F.col("order_delivered_customer_date").isNotNull()
    )
).select(
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
).show(30, truncate=False)

+--------------------------------+------------+------------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |order_status|order_purchase_timestamp|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+------------+------------------------+----------------------------+-----------------------------+-----------------------------+
|58d4c4747ee059eeeb865b349b41f53a|delivered   |2018-07-21 12:49:32     |NULL                        |NULL                         |2018-07-31 00:00:00          |
|4df92d82d79c3b52c7138679fa9b07fc|delivered   |2018-07-24 11:32:11     |NULL                        |NULL                         |2018-08-06 00:00:00          |
|6e57e23ecac1ae881286657694444267|delivered   |2018-08-09 17:36:47     |NULL                        |NULL                         |2018-09-06 00:00:00          |
|1950d777989f6a877539f53795b

In [0]:
from pyspark.sql import functions as F

orders_base = spark.table("workspace.silver.silver_orders")

fact_orders_df = (
    orders_base
    .withColumn(
        "purchase_date_key",
        F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int")
    )
    .withColumn(
        "delivery_date_key",
        F.when(
            F.col("order_delivered_customer_date").isNotNull(),
            F.date_format(
                "order_delivered_customer_date", "yyyyMMdd"
            ).cast("int")
        )
    )
    .withColumn(
        "delivery_days_from_purchase",
        F.when(
            F.col("order_delivered_customer_date").isNotNull(),
            F.datediff(
                F.to_date("order_delivered_customer_date"),
                F.to_date("order_purchase_timestamp")
            )
        )
    )
    .withColumn(
        "approval_to_delivery_days",
        F.when(
            F.col("order_approved_at").isNotNull() &
            F.col("order_delivered_customer_date").isNotNull(),
            F.datediff(
                F.to_date("order_delivered_customer_date"),
                F.to_date("order_approved_at")
            )
        )
    )
    .withColumn(
        "delivery_delay_days",
        F.when(
            (F.col("order_status") == "delivered") &
            F.col("order_delivered_customer_date").isNotNull(),
            F.datediff(
                F.to_date("order_delivered_customer_date"),
                F.to_date("order_estimated_delivery_date")
            )
        )
    )
    .withColumn(
        "delivery_performance",
        F.when(
            F.col("order_status") != "delivered",
            "not_applicable"
        ).when(
            F.col("order_delivered_customer_date").isNull(),
            "unknown"
        ).when(
            F.col("delivery_delay_days") > 0,
            "late"
        ).otherwise("on_time_or_early")
    )
    .withColumn(
        "is_delivered",
        (F.col("order_status") == "delivered").cast("int")
    )
    .withColumn(
        "is_cancelled",
        (F.col("order_status") == "canceled").cast("int")
    )
)

print("Fact orders rows:", fact_orders_df.count())
print("Distinct order IDs:", fact_orders_df.select("order_id").distinct().count())

print("\nDelivery performance breakdown:")
fact_orders_df.groupBy("delivery_performance").count().orderBy(
    "delivery_performance"
).show()

print("\nSample records:")
display(
    fact_orders_df.select(
        "order_id",
        "order_status",
        "delivery_days_from_purchase",
        "delivery_delay_days",
        "delivery_performance",
        "is_delivered",
        "is_cancelled"
    ).limit(10)
)

Fact orders rows: 99441
Distinct order IDs: 99441

Delivery performance breakdown:
+--------------------+-----+
|delivery_performance|count|
+--------------------+-----+
|                late| 6533|
|      not_applicable| 2963|
|    on_time_or_early|89853|
|             unknown|   92|
+--------------------+-----+


Sample records:


order_id,order_status,delivery_days_from_purchase,delivery_delay_days,delivery_performance,is_delivered,is_cancelled
e481f51cbdc54678b7cc49136f2d6af7,delivered,8,-8,on_time_or_early,1,0
53cdb2fc8bc7dce0b6741e2150273451,delivered,14,-6,on_time_or_early,1,0
47770eb9100c2d0c44946d9cf07ec65d,delivered,9,-18,on_time_or_early,1,0
949d5b44dbf5de918fe9c16f97b45f8a,delivered,14,-13,on_time_or_early,1,0
ad21c59c0840e6cb83a9ceb5573f8159,delivered,3,-10,on_time_or_early,1,0
a4591c265e18cb1dcee52889e2d8acc3,delivered,17,-6,on_time_or_early,1,0
136cce7faa42fdb2cefd53fdc79a6098,invoiced,null,null,not_applicable,0,0
6514b8ad8028c9f2cc2374ded245783f,delivered,10,-12,on_time_or_early,1,0
76c6e866289321a7c93b82b54852dc33,delivered,10,-32,on_time_or_early,1,0
e69bfb5eb88e0ed6a785585b27e16dbf,delivered,18,-7,on_time_or_early,1,0


In [0]:
# Save the order-level fact table in the Gold schema
fact_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.fact_orders")

# Verify the saved table
saved_fact_orders = spark.table("workspace.gold.fact_orders")

print("Saved table: workspace.gold.fact_orders")
print("Total rows:", saved_fact_orders.count())
print("Distinct orders:", saved_fact_orders.select("order_id").distinct().count())

print("\nDelivery performance:")
saved_fact_orders.groupBy("delivery_performance") \
    .count() \
    .orderBy("delivery_performance") \
    .show()

Saved table: workspace.gold.fact_orders
Total rows: 99441
Distinct orders: 99441

Delivery performance:
+--------------------+-----+
|delivery_performance|count|
+--------------------+-----+
|                late| 6533|
|      not_applicable| 2963|
|    on_time_or_early|89853|
|             unknown|   92|
+--------------------+-----+



In [0]:
from pyspark.sql import functions as F

fact_orders = spark.table("workspace.gold.fact_orders")

kpi_monthly_delivery = (
    fact_orders
    .filter(F.col("order_status") == "delivered")
    .groupBy(
        F.date_format(
            "order_purchase_timestamp", "yyyy-MM"
        ).alias("purchase_month")
    )
    .agg(
        F.count("*").alias("delivered_orders"),
        F.sum(
            F.when(
                F.col("delivery_performance") == "on_time_or_early", 1
            ).otherwise(0)
        ).alias("on_time_or_early_orders"),
        F.sum(
            F.when(
                F.col("delivery_performance") == "late", 1
            ).otherwise(0)
        ).alias("late_orders"),
        F.sum(
            F.when(
                F.col("delivery_performance") == "unknown", 1
            ).otherwise(0)
        ).alias("unknown_delivery_orders"),
        F.round(
            F.avg(
                F.when(
                    F.col("delivery_performance") != "unknown",
                    F.col("delivery_delay_days")
                )
            ), 2
        ).alias("average_delivery_delay_days"),
        F.round(
            F.avg(
                F.when(
                    F.col("delivery_performance") == "on_time_or_early", 1
                ).when(
                    F.col("delivery_performance") == "late", 0
                )
            ) * 100, 2
        ).alias("on_time_delivery_percentage")
    )
    .orderBy("purchase_month")
)

display(kpi_monthly_delivery)

purchase_month,delivered_orders,on_time_or_early_orders,late_orders,unknown_delivery_orders,average_delivery_delay_days,on_time_delivery_percentage
2016-09,1,0,1,0,36.0,0.0
2016-10,265,259,2,4,-36.63,99.23
2016-12,1,1,0,0,-22.0,100.0
2017-01,750,727,22,1,-27.36,97.06
2017-02,1653,1601,49,3,-19.19,97.03
2017-03,2546,2429,116,1,-12.33,95.44
2017-04,2303,2152,151,0,-12.97,93.44
2017-05,3546,3438,106,2,-13.5,97.01
2017-06,3135,3038,95,2,-12.64,96.97
2017-07,3872,3753,108,11,-12.45,97.2


In [0]:
# Use a clearer business-friendly metric name
kpi_monthly_delivery_final = kpi_monthly_delivery.withColumnRenamed(
    "average_delivery_delay_days",
    "average_days_vs_estimated_delivery"
)

# Save to Gold
kpi_monthly_delivery_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.kpi_monthly_delivery")

# Verify saved table and reconciliation
saved_delivery_kpi = spark.table("workspace.gold.kpi_monthly_delivery")

saved_delivery_kpi.agg(
    F.count("*").alias("months"),
    F.sum("delivered_orders").alias("total_delivered"),
    F.sum("on_time_or_early_orders").alias("total_on_time_or_early"),
    F.sum("late_orders").alias("total_late"),
    F.sum("unknown_delivery_orders").alias("total_unknown")
).show()

print("Saved successfully: workspace.gold.kpi_monthly_delivery")

+------+---------------+----------------------+----------+-------------+
|months|total_delivered|total_on_time_or_early|total_late|total_unknown|
+------+---------------+----------------------+----------+-------------+
|    23|          96478|                 89853|      6533|           92|
+------+---------------+----------------------+----------+-------------+

Saved successfully: workspace.gold.kpi_monthly_delivery


In [0]:
fact_sales_df = spark.table("workspace.gold.fact_sales")

print("Total sales records:", fact_sales_df.count())
print("Total columns:", len(fact_sales_df.columns))

fact_sales_df.printSchema()

display(fact_sales_df.limit(5))

Total sales records: 112650
Total columns: 28
root
 |-- purchase_date: date (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)


purchase_date,order_id,order_item_id,customer_id,customer_unique_id,product_id,product_category_name_english,seller_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,shipping_limit_date,price,freight_value,customer_zip_code_prefix,customer_city,customer_state,seller_zip_code_prefix,seller_city,seller_state,purchase_date_key,customer_geo_key,seller_geo_key,total_item_value,order_item_count
2018-04-25,00335f75ea6a4455b524a8dd177067f5,1,32b84239e3f3904ac162d832454770c7,a666bec8af192f2238a0b719a7063a00,e1bf2f209fd2b08e1af3ed71732d9008,pet_shop,70c27847eca8195c983ed7e798c56743,delivered,2018-04-25T21:47:26.000Z,2018-04-25T22:10:29.000Z,2018-04-27T14:49:00.000Z,2018-05-03T22:16:51.000Z,2018-05-28T00:00:00.000Z,2018-05-02T22:10:29.000Z,168.8,8.21,23093,rio de janeiro,RJ,20930,rio de janeiro,RJ,20180425,23093,20930,177.01,1
2018-02-02,00378c6c981f234634c0b9d6128df6dd,1,f3476df115c5d26bdb6bfb71b5fb66af,cd2fd2a38f5903eb2a7377b548e0bfdd,38fa750a3a3b3204f169c86a3284d387,sports_leisure,218d46b86c1881d022bce9c68a7d4b15,delivered,2018-02-02T19:39:46.000Z,2018-02-02T19:50:27.000Z,2018-02-05T18:06:49.000Z,2018-02-26T21:36:00.000Z,2018-02-26T00:00:00.000Z,2018-02-08T19:50:27.000Z,41.0,11.85,04826,sao paulo,SP,14070,ribeirao preto,SP,20180202,4826,14070,52.85,1
2017-12-25,004ba47bb589cdbc485e6025bd6d3c81,1,ab30bf72cae845f3814c7baa50ca549a,8b9c297f4874622ea42c249bba3854d7,8c5876b1c7768217964f353bc7e64393,telephony,0db783cfcd3b73998abc6e10e59a102f,delivered,2017-12-25T20:40:15.000Z,2017-12-25T20:48:10.000Z,2017-12-26T20:39:05.000Z,2017-12-29T20:24:23.000Z,2018-01-19T00:00:00.000Z,2017-12-29T20:48:10.000Z,49.0,11.85,15775,santa fe do sul,SP,11010,santos,SP,20171225,15775,11010,60.85,1
2017-11-24,007ec946d4f798caf2b43c9d44c56421,1,3ace63287bc8b41172016c06a5093808,25e6bea0f352b9c61d1bed324da0f5cb,779559842fc122d55edbd03153f35e81,sports_leisure,f8db351d8c4c4c22c6835c19a46f01b0,delivered,2017-11-24T13:05:15.000Z,2017-11-25T04:15:38.000Z,2017-11-28T19:27:06.000Z,2017-12-06T22:38:40.000Z,2017-12-26T00:00:00.000Z,2017-11-30T04:15:38.000Z,27.9,15.1,95670,gramado,RS,13324,salto,SP,20171124,95670,13324,43.0,1
2018-05-11,008990da719764fb392914d8ab53ad04,1,cc4e2f483459498c9ce82a70ff415752,d3391cadc57741a0defe8a569e476813,a8115d34a4b7b74388da609692162c37,computers_accessories,85d9eb9ddc5d00ca9336a2219c97bb13,delivered,2018-05-11T19:20:54.000Z,2018-05-11T19:37:41.000Z,2018-05-24T13:47:00.000Z,2018-06-05T14:18:36.000Z,2018-06-19T00:00:00.000Z,2018-06-01T19:37:41.000Z,35.9,18.23,27930,macae,RJ,31255,belo horizonte,MG,20180511,27930,31255,54.13,1


In [0]:
from pyspark.sql import functions as F

# First: one row per customer per order
customer_orders = (
    fact_sales_df
    .groupBy("customer_unique_id", "order_id")
    .agg(
        F.sum("price").alias("order_product_value"),
        F.sum("freight_value").alias("order_freight_value"),
        F.sum("total_item_value").alias("order_total_value"),
        F.countDistinct("product_id").alias("distinct_products"),
        F.min("purchase_date").alias("purchase_date"),
        F.first("customer_state").alias("customer_state")
    )
)

# Second: aggregate orders at customer level
customer_behavior_df = (
    customer_orders
    .groupBy("customer_unique_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("order_product_value").alias("total_product_spend"),
        F.sum("order_freight_value").alias("total_freight_spend"),
        F.sum("order_total_value").alias("total_spend"),
        F.avg("order_total_value").alias("average_order_value"),
        F.min("purchase_date").alias("first_purchase_date"),
        F.max("purchase_date").alias("last_purchase_date"),
        F.sum("distinct_products").alias("total_distinct_products"),
        F.first("customer_state").alias("customer_state")
    )
    .withColumn(
        "customer_type",
        F.when(F.col("total_orders") > 1, "repeat_customer")
         .otherwise("one_time_customer")
    )
    .withColumn(
        "customer_lifetime_days",
        F.datediff("last_purchase_date", "first_purchase_date")
    )
)

print("Customer records:", customer_behavior_df.count())
print("Unique customers:", customer_behavior_df.select("customer_unique_id").distinct().count())

print("\nCustomer type distribution:")
customer_behavior_df.groupBy("customer_type").agg(
    F.count("*").alias("customers"),
    F.round(F.avg("total_spend"), 2).alias("avg_customer_spend"),
    F.round(F.avg("total_orders"), 2).alias("avg_orders_per_customer")
).show()

display(customer_behavior_df.limit(10))

Customer records: 95420
Unique customers: 95420

Customer type distribution:
+-----------------+---------+------------------+-----------------------+
|    customer_type|customers|avg_customer_spend|avg_orders_per_customer|
+-----------------+---------+------------------+-----------------------+
|one_time_customer|    92507|            161.49|                    1.0|
|  repeat_customer|     2913|            310.49|                   2.11|
+-----------------+---------+------------------+-----------------------+



customer_unique_id,total_orders,total_product_spend,total_freight_spend,total_spend,average_order_value,first_purchase_date,last_purchase_date,total_distinct_products,customer_state,customer_type,customer_lifetime_days
a666bec8af192f2238a0b719a7063a00,1,168.8,8.21,177.01,177.01,2018-04-25,2018-04-25,1,RJ,one_time_customer,0
cd2fd2a38f5903eb2a7377b548e0bfdd,1,41.0,11.85,52.85,52.85,2018-02-02,2018-02-02,1,SP,one_time_customer,0
8b9c297f4874622ea42c249bba3854d7,1,49.0,11.85,60.85,60.85,2017-12-25,2017-12-25,1,SP,one_time_customer,0
25e6bea0f352b9c61d1bed324da0f5cb,1,27.9,15.1,43.0,43.0,2017-11-24,2017-11-24,1,RS,one_time_customer,0
d3391cadc57741a0defe8a569e476813,1,35.9,18.23,54.13,54.13,2018-05-11,2018-05-11,1,RJ,one_time_customer,0
1d74273c053909be057f50f65b48522f,1,199.8,32.92,232.72,232.72,2017-12-09,2017-12-09,2,SP,one_time_customer,0
383fa1a759c0437605732a46d4f43d2c,1,24.89,7.75,32.64,32.64,2018-07-25,2018-07-25,1,DF,one_time_customer,0
15b521471c36ed411359347ff8257b79,1,50.0,21.19,71.19,71.19,2017-08-02,2017-08-02,1,SP,one_time_customer,0
b9b70004b9682db7a07defb7ab1b475d,1,79.0,13.91,92.91,92.91,2018-04-20,2018-04-20,1,SP,one_time_customer,0
a67fecd4801b668a6f99946a13333c61,1,149.0,40.37,189.37,189.37,2018-01-30,2018-01-30,1,RJ,one_time_customer,0


In [0]:
# Validate customer-level aggregation
print("Customer-level validation:")

customer_behavior_df.agg(
    F.count("*").alias("unique_customers"),
    F.sum("total_orders").alias("total_orders"),
    F.round(F.sum("total_product_spend"), 2).alias("product_sales"),
    F.round(F.sum("total_freight_spend"), 2).alias("freight_sales"),
    F.round(F.sum("total_spend"), 2).alias("total_sales")
).show()

print("Original fact_sales totals:")

fact_sales_df.agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.round(F.sum("price"), 2).alias("product_sales"),
    F.round(F.sum("freight_value"), 2).alias("freight_sales"),
    F.round(F.sum("total_item_value"), 2).alias("total_sales")
).show()

print("Customer type counts:")
customer_behavior_df.groupBy("customer_type").count().show()

Customer-level validation:
+----------------+------------+-------------+-------------+-------------+
|unique_customers|total_orders|product_sales|freight_sales|  total_sales|
+----------------+------------+-------------+-------------+-------------+
|           95420|       98666| 1.35916437E7|   2251909.54|1.584355324E7|
+----------------+------------+-------------+-------------+-------------+

Original fact_sales totals:
+------------+-------------+-------------+-------------+
|total_orders|product_sales|freight_sales|  total_sales|
+------------+-------------+-------------+-------------+
|       98666| 1.35916437E7|   2251909.54|1.584355324E7|
+------------+-------------+-------------+-------------+

Customer type counts:
+-----------------+-----+
|    customer_type|count|
+-----------------+-----+
|one_time_customer|92507|
|  repeat_customer| 2913|
+-----------------+-----+



In [0]:
# Save customer-level behaviour as a Gold Delta table
customer_behavior_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.agg_customer_behavior")

# Verify the saved table
saved_customer_behavior = spark.table(
    "workspace.gold.agg_customer_behavior"
)

print("Saved table: workspace.gold.agg_customer_behavior")
print("Total customer records:", saved_customer_behavior.count())

print("\nCustomer segment verification:")
saved_customer_behavior.groupBy("customer_type").agg(
    F.count("*").alias("customers"),
    F.round(
        F.count("*") * 100.0 / saved_customer_behavior.count(), 2
    ).alias("customer_percentage"),
    F.round(F.avg("total_spend"), 2).alias("average_customer_spend")
).show()

Saved table: workspace.gold.agg_customer_behavior
Total customer records: 95420

Customer segment verification:
+-----------------+---------+-------------------+----------------------+
|    customer_type|customers|customer_percentage|average_customer_spend|
+-----------------+---------+-------------------+----------------------+
|one_time_customer|    92507|              96.95|                161.49|
|  repeat_customer|     2913|               3.05|                310.49|
+-----------------+---------+-------------------+----------------------+



In [0]:
from pyspark.sql import functions as F

customer_df = spark.table("workspace.gold.agg_customer_behavior")

total_customers = customer_df.count()
total_revenue = customer_df.agg(
    F.sum("total_spend")
).first()[0]

kpi_customer_segments = (
    customer_df
    .groupBy("customer_type")
    .agg(
        F.count("*").alias("customer_count"),
        F.sum("total_orders").alias("total_orders"),
        F.round(F.avg("total_orders"), 2).alias("avg_orders_per_customer"),
        F.round(F.avg("total_spend"), 2).alias("avg_customer_spend"),
        F.round(F.avg("average_order_value"), 2).alias("avg_order_value"),
        F.round(F.sum("total_spend"), 2).alias("segment_revenue"),
        F.round(F.avg("customer_lifetime_days"), 2).alias(
            "avg_customer_lifetime_days"
        )
    )
    .withColumn(
        "customer_percentage",
        F.round(F.col("customer_count") * 100 / total_customers, 2)
    )
    .withColumn(
        "revenue_percentage",
        F.round(F.col("segment_revenue") * 100 / total_revenue, 2)
    )
    .orderBy("customer_type")
)

display(kpi_customer_segments)

customer_type,customer_count,total_orders,avg_orders_per_customer,avg_customer_spend,avg_order_value,segment_revenue,avg_customer_lifetime_days,customer_percentage,revenue_percentage
one_time_customer,92507,92507,1.0,161.49,161.49,1.493910699E7,0.0,96.95,94.29
repeat_customer,2913,6159,2.11,310.49,146.85,904446.25,87.9,3.05,5.71


In [0]:
# Save customer segment KPIs
kpi_customer_segments.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.kpi_customer_segments")

# Verify saved table
saved_segments = spark.table("workspace.gold.kpi_customer_segments")

print("Saved table: workspace.gold.kpi_customer_segments")

saved_segments.agg(
    F.count("*").alias("customer_segments"),
    F.sum("customer_count").alias("total_customers"),
    F.sum("total_orders").alias("total_orders"),
    F.round(F.sum("segment_revenue"), 2).alias("total_revenue"),
    F.sum("customer_percentage").alias("customer_percentage_total"),
    F.sum("revenue_percentage").alias("revenue_percentage_total")
).show()

display(saved_segments)

Saved table: workspace.gold.kpi_customer_segments
+-----------------+---------------+------------+-------------+-------------------------+------------------------+
|customer_segments|total_customers|total_orders|total_revenue|customer_percentage_total|revenue_percentage_total|
+-----------------+---------------+------------+-------------+-------------------------+------------------------+
|                2|          95420|       98666|1.584355324E7|                    100.0|                   100.0|
+-----------------+---------------+------------+-------------+-------------------------+------------------------+



customer_type,customer_count,total_orders,avg_orders_per_customer,avg_customer_spend,avg_order_value,segment_revenue,avg_customer_lifetime_days,customer_percentage,revenue_percentage
one_time_customer,92507,92507,1.0,161.49,161.49,1.493910699E7,0.0,96.95,94.29
repeat_customer,2913,6159,2.11,310.49,146.85,904446.25,87.9,3.05,5.71


In [0]:
from pyspark.sql import functions as F

customer_df = spark.table("workspace.gold.agg_customer_behavior")

kpi_monthly_customer_acquisition = (
    customer_df
    .groupBy(
        F.date_format(
            "first_purchase_date", "yyyy-MM"
        ).alias("acquisition_month")
    )
    .agg(
        F.count("*").alias("new_customers"),
        F.sum(
            F.when(F.col("customer_type") == "one_time_customer", 1)
             .otherwise(0)
        ).alias("one_time_customers"),
        F.sum(
            F.when(F.col("customer_type") == "repeat_customer", 1)
             .otherwise(0)
        ).alias("repeat_customers"),
        F.round(F.avg("total_spend"), 2).alias("avg_customer_lifetime_value"),
        F.round(F.sum("total_spend"), 2).alias("cohort_lifetime_revenue")
    )
    .withColumn(
        "repeat_customer_percentage",
        F.round(
            F.col("repeat_customers") * 100 /
            F.col("new_customers"), 2
        )
    )
    .orderBy("acquisition_month")
)

display(kpi_monthly_customer_acquisition)

acquisition_month,new_customers,one_time_customers,repeat_customers,avg_customer_lifetime_value,cohort_lifetime_revenue,repeat_customer_percentage
2016-09,3,3,0,118.25,354.75,0.0
2016-10,305,293,12,192.0,58559.53,3.93
2016-12,1,0,1,39.24,39.24,100.0
2017-01,754,699,55,186.58,140681.1,7.29
2017-02,1705,1635,70,171.84,292994.87,4.11
2017-03,2595,2466,129,172.42,447426.75,4.97
2017-04,2339,2232,107,181.97,425636.17,4.57
2017-05,3560,3369,191,169.61,603805.72,5.37
2017-06,3114,2944,170,166.71,519131.75,5.46
2017-07,3843,3662,181,154.65,594319.81,4.71


In [0]:
# Save monthly customer acquisition KPI
kpi_monthly_customer_acquisition.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.kpi_monthly_customer_acquisition")

# Verify saved table
saved_acquisition = spark.table(
    "workspace.gold.kpi_monthly_customer_acquisition"
)

print("Saved table: workspace.gold.kpi_monthly_customer_acquisition")

saved_acquisition.agg(
    F.count("*").alias("months"),
    F.sum("new_customers").alias("total_customers"),
    F.sum("one_time_customers").alias("one_time_customers"),
    F.sum("repeat_customers").alias("repeat_customers"),
    F.round(
        F.sum("cohort_lifetime_revenue"), 2
    ).alias("total_cohort_revenue")
).show()

display(saved_acquisition.orderBy("acquisition_month"))

Saved table: workspace.gold.kpi_monthly_customer_acquisition
+------+---------------+------------------+----------------+--------------------+
|months|total_customers|one_time_customers|repeat_customers|total_cohort_revenue|
+------+---------------+------------------+----------------+--------------------+
|    23|          95420|             92507|            2913|       1.584355324E7|
+------+---------------+------------------+----------------+--------------------+



acquisition_month,new_customers,one_time_customers,repeat_customers,avg_customer_lifetime_value,cohort_lifetime_revenue,repeat_customer_percentage
2016-09,3,3,0,118.25,354.75,0.0
2016-10,305,293,12,192.0,58559.53,3.93
2016-12,1,0,1,39.24,39.24,100.0
2017-01,754,699,55,186.58,140681.1,7.29
2017-02,1705,1635,70,171.84,292994.87,4.11
2017-03,2595,2466,129,172.42,447426.75,4.97
2017-04,2339,2232,107,181.97,425636.17,4.57
2017-05,3560,3369,191,169.61,603805.72,5.37
2017-06,3114,2944,170,166.71,519131.75,5.46
2017-07,3843,3662,181,154.65,594319.81,4.71


In [0]:
# List every table in the Gold schema
gold_tables = spark.sql("SHOW TABLES IN workspace.gold")

print("Total Gold tables:", gold_tables.count())

display(
    gold_tables
    .select("tableName")
    .orderBy("tableName")
)

Total Gold tables: 26


tableName
agg_customer_behavior
dim_customers
dim_date
dim_geolocation
dim_products
dim_sellers
fact_marketing_conversion
fact_orders
fact_payments
fact_reviews
